<a href="https://colab.research.google.com/github/mariaantipina/PythonPractice/blob/main/Antipina_Matplotlib_Practica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Вариант 13. Фитнес-клуб (Matplotlib + Seaborn). Полный курс, 60 задач

---

## 🟢 Уровень 1. Новичок (задачи 1–12)

**Задача 1 (первый график, Figure, Axes, savefig).** Фитнес-клуб ведёт учёт ежедневной посещаемости — данные в `visits.csv` со столбцами `date` и `visits`. Загрузите через `pd.read_csv("visits.csv", parse_dates=["date"], index_col="date")`, создайте `fig, ax = plt.subplots()`, постройте `ax.plot(df.index, df["visits"])` и сохраните `visits.png` с `dpi=150`, `bbox_inches="tight"`. Выведите `type(fig)`, `type(ax)`, `df.index.dtype`. Ожидаемый вывод: `Figure`, `Axes`, `datetime64[ns]`; файл создан. В комментариях объясните: чем `fig` отличается от `ax` и почему это разделение принципиально для сложных фигур (один холст — несколько графиков, suptitle, colorbar); что делает `plt.subplots()` «под капотом», если параметры не указаны (`Figure.add_subplot(111)`); почему `savefig` нужно вызывать **до** `plt.show()` (фигура может быть очищена и сохранится пустой PNG); зачем нужен `bbox_inches="tight"` и что произойдёт с длинными подписями без него; что случится, если забыть `parse_dates` при чтении CSV — как будут выглядеть даты по оси X (числа дней с 1970 года или категории вместо дат); почему `x` и `y` должны быть одинаковой длины и какую ошибку выбросит Matplotlib при несовпадении (`ValueError: x and y must have same first dimension`).


In [ ]:
import pandas as pd
import numpy as np

# Генерируем тестовый visits.csv
np.random.seed(42)
dates = pd.date_range("2024-01-01", periods=90, freq="D")
visits = np.random.randint(80, 250, size=90)

df_src = pd.DataFrame({"date": dates, "visits": visits})
df_src.to_csv("visits.csv", index=False)

print("visits.csv создан:", df_src.shape)
print(df_src.head())
print()

In [ ]:
import matplotlib.pyplot as plt
df = pd.read_csv("visits.csv", parse_dates=["date"], index_col="date")

# Создание фигуры и осей
fig, ax = plt.subplots()
ax.plot(df.index, df["visits"])

# Сохранение до show()
fig.savefig("visits.png", dpi=150, bbox_inches="tight")

# Проверки
print(type(fig).__name__)
print(type(ax).__name__)
print(df.index.dtype)

# Комментарии:
# fig — весь холст (Figure), ax — конкретная область с графиком (Axes).
# Разделение принципиально: одна Figure может содержать много Axes
# (subplots), общий suptitle, colorbar и т.д.
# plt.subplots() без аргументов по сути делает Figure.add_subplot(111) —
# одна фигура, одна ось.
# savefig нужно вызывать ДО plt.show(): после show() фигура может быть
# очищена, и сохранится пустой PNG.
# bbox_inches="tight" обрезает лишние поля и гарантирует, что длинные
# подписи/легенды не обрежутся; без него они могут выйти за границы.
# Без parse_dates даты останутся строками (или будут интерпретированы
# как числа — дни с 1970), ось X будет некорректной.
# x и y должны быть одинаковой длины, иначе Matplotlib выбросит
# ValueError: x and y must have same first dimension.

plt.show()
print()



**Задача 2 (параметры figure, PlotConfig).** Фитнес-клуб готовит серию отчётов для управляющей компании сети, и нужно, чтобы все графики имели одинаковый стиль. Определите `@dataclass class PlotConfig` с полями `figsize: tuple = (10, 6)`, `dpi: int = 100`, `facecolor: str = "white"`, `layout: str = "constrained"` и реализуйте фабрику `make_figure(cfg) -> tuple[Figure, Axes]`. Постройте график посещаемости с конфигурацией `PlotConfig(figsize=(12, 8), dpi=200, facecolor="#f5f5f5")` и выведите итоговый размер в пикселях. Ожидаемый вывод: 2400 × 1600 пикселей. В комментариях объясните: как `figsize` и `dpi` связаны с размером в пикселях и почему `figsize` в дюймах, а не в пикселях; что даёт `layout="constrained"` и почему он рекомендуется вместо `tight_layout`; чем `facecolor` фигуры отличается от `ax.set_facecolor`; зачем нужны разные `dpi` для веба (100–150) и для печати (300+); как расширить датакласс методом `with_dpi(new_dpi)` через `dataclasses.replace` и почему это функциональный подход поверх ООП; что произойдёт, если задать `facecolor` фигуры и осей разными — где это может быть полезно в презентации фитнес-клуба (белый фон осей на фирменном оранжевом).

In [ ]:
from dataclasses import dataclass, replace
from typing import Tuple
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.axes import Axes

@dataclass
class PlotConfig:
    figsize: Tuple[float, float] = (10, 6)
    dpi: int = 100
    facecolor: str = "white"
    layout: str = "constrained"

    def with_dpi(self, new_dpi: int) -> "PlotConfig":
        return replace(self, dpi=new_dpi)

def make_figure(cfg: PlotConfig) -> Tuple[Figure, Axes]:
    fig, ax = plt.subplots(
        figsize=cfg.figsize,
        dpi=cfg.dpi,
        facecolor=cfg.facecolor,
        layout=cfg.layout,
    )
    return fig, ax

# Загрузка данных
df = pd.read_csv("visits.csv", parse_dates=["date"], index_col="date")

# Конфигурация из задания
cfg = PlotConfig(figsize=(12, 8), dpi=200, facecolor="#f5f5f5")
fig, ax = make_figure(cfg)
ax.plot(df.index, df["visits"])
ax.set_title("Посещаемость фитнес-клуба")

# Размер в пикселях
w_px = int(cfg.figsize[0] * cfg.dpi)
h_px = int(cfg.figsize[1] * cfg.dpi)
print(f"{w_px} × {h_px} пикселей")

# Демонстрация with_dpi
cfg_web = cfg.with_dpi(100)
print("Исходный dpi:", cfg.dpi, "| web dpi:", cfg_web.dpi)

fig.savefig("visits_styled.png", dpi=cfg.dpi, bbox_inches="tight")

# Комментарии:
# Размер в пикселях = figsize (дюймы) × dpi. figsize в дюймах, потому что
# это физический размер при печати — независим от разрешения устройства.
# layout="constrained" автоматически разводит подписи/титулы, не требуя
# ручных отступов; рекомендуется вместо tight_layout, т.к. учитывает
# colorbar, suptitle, вложенные оси и корректно работает при resize.
# facecolor фигуры — фон всего холста; ax.set_facecolor — фон только
# области данных. Разные цвета (белый ax на фирменном оранжевом fig)
# выглядят презентационно: график "парит" на брендовом фоне.
# dpi: 100–150 для веба (быстрая загрузка, экраны), 300+ для печати
# (качество полиграфии).
# with_dpi через dataclasses.replace создаёт НОВЫЙ объект — это
# функциональный подход (иммутабельность) поверх ООП: конфиг можно
# безопасно шарить между отчётами, не боясь мутаций.
# Если facecolor fig и ax различаются — получаем рамку-подложку бренда.

plt.show()
print()



**Задача 3 (метки, заголовок, WeightFormatter).** Фитнес-клуб анализирует средний вес клиентов по месяцам, и на оси Y нужен формат `78.5 кг` с точностью до десятых. Реализуйте `class WeightFormatter(mticker.Formatter)` с методом `__call__(self, x, pos=None)`, возвращающим `f"{x:.1f} кг"`, и примените через `ax.yaxis.set_major_formatter(WeightFormatter())`. Установите `xlabel("Месяц")`, `ylabel("Средний вес")`, `title("Средний вес клиентов по месяцам")`, для заголовка — `fontweight="bold"`, `fontfamily="serif"`, для оси X — `labelpad=15`. В комментариях объясните: как `Formatter` встраивается в ООП-иерархию Matplotlib и почему наследование от `mticker.Formatter` даёт больше возможностей, чем функция в `FuncFormatter`; в каких случаях `Formatter`-класс хранит состояние (единица измерения, точность, суффикс); почему `fontfamily="serif"` может дать разный шрифт на разных ОС и как это влияет на воспроизводимость отчёта; что делает `labelpad` и когда его нужно увеличивать; как `loc="left"` в `set_title` смещает заголовок к левому краю; зачем использовать `fontdict` вместо именованных аргументов, если одинаковые настройки применяются к нескольким текстовым элементам.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- данные ---
# Агрегируем синтетический вес по месяцам из уже существующего df
# (df был загружен в задаче 1; если запуск отдельный — перечитаем)
try:
    df
except NameError:
    df = pd.read_csv("visits.csv", parse_dates=["date"], index_col="date")

# Синтетический средний вес по месяцам
months = pd.Series(df.index.month, index=df.index)
weight_month = pd.Series(
    70 + 10 * (months - 1) / 11 + (months % 3) * 0.4,
    index=df.index,
).resample("ME").mean()

# --- форматтер ---
class WeightFormatter(mticker.Formatter):
    def __call__(self, x, pos=None):
        return f"{x:.1f} кг"

fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
ax.plot(weight_month.index, weight_month.values, marker="o")

ax.yaxis.set_major_formatter(WeightFormatter())
ax.set_xlabel("Месяц", labelpad=15)
ax.set_ylabel("Средний вес")
ax.set_title(
    "Средний вес клиентов по месяцам",
    fontweight="bold",
    fontfamily="serif",
    loc="left",
)

# Комментарии:
# Formatter — часть ООП-иерархии Matplotlib: ось вызывает __call__(x, pos)
# для каждого тика. Наследование от mticker.Formatter даёт полноценный
# класс (можно хранить состояние, переопределять set_locs, использовать
# в нескольких осях), тогда как FuncFormatter — это лишь обёртка над
# лямбдой, без состояния и с меньшими возможностями настройки.
# Класс-Formatter хранит состояние: единицу измерения (кг/фунты),
# точность (1 знак), суффикс — удобно для переиспользования и локализации.
# fontfamily="serif" даёт разные физические шрифты на разных ОС
# (DejaVu Serif / Times / Cambria), что ломает воспроизводимость отчёта;
# в продакшене задают явный шрифт (rcParams["font.family"]).
# labelpad — отступ подписи оси от тиков; увеличивают, когда подписи
# тиков длинные (например, "78.5 кг") и начинают наезжать на заголовок.
# loc="left" в set_title выравнивает заголовок по левому краю оси —
# типичный стиль деловых отчётов.
# fontdict удобен, когда одинаковый набор (size, weight, color) применяется
# к нескольким текстам — задаём один раз и переиспользуем, избегая дублей.

plt.show()
print()


**Задача 4 (стили линий, LineStyle).** Фитнес-клуб строит график сравнения трёх форматов занятий (йога, кроссфит, бассейн) на одной оси — число посещений по месяцам. Чтобы не дублировать десятки аргументов в каждом `plot`, создайте `@dataclass class LineStyle` с полями `color: str = "steelblue"`, `linewidth: float = 2`, `marker: str = "o"`, `markersize: int = 8`, `alpha: float = 1` и методом `as_kwargs() -> dict`. Постройте 5 подграфиков с разными `LineStyle`, на двух из них покажите эквивалентность строки `"ob-"` и именованных аргументов `color="b", marker="o", linestyle="-"`. Ожидаемый вывод: 5 линий с разными маркерами и цветами. В комментариях объясните: как читается строка формата `"[marker][line][color]"` и почему её иногда удобнее именованных аргументов; чем `markerfacecolor` отличается от `markeredgecolor` и когда их стоит различать; как `alpha < 1` улучшает читаемость при наложении нескольких линий; почему `linewidth` измеряется в точках, а не в пикселях, и как это связано с `dpi`; что произойдёт при `alpha=0` и как это можно использовать для «скрытых» линий; как задать цвет HEX-кодом или RGB-кортежем для соответствия брендбуку клуба.

In [ ]:
from dataclasses import dataclass, field
import pandas as pd
import matplotlib.pyplot as plt

@dataclass
class LineStyle:
    color: str = "steelblue"
    linewidth: float = 2
    marker: str = "o"
    markersize: int = 8
    alpha: float = 1

    def as_kwargs(self) -> dict:
        return {
            "color": self.color,
            "linewidth": self.linewidth,
            "marker": self.marker,
            "markersize": self.markersize,
            "alpha": self.alpha,
        }

# --- данные ---
try:
    df
except NameError:
    df = pd.read_csv("visits.csv", parse_dates=["date"], index_col="date")

months = pd.date_range("2024-01-01", periods=6, freq="MS")
yoga    = [120, 135, 150, 140, 160, 175]
crossfit= [ 90, 110, 130, 155, 170, 190]
pool    = [ 80,  95, 100, 120, 125, 140]

styles = [
    LineStyle(color="steelblue", marker="o"),
    LineStyle(color="crimson",   marker="s", linewidth=2.5),
    LineStyle(color="seagreen",  marker="^", markersize=10),
    LineStyle(color="b", marker="o", alpha=0.6),          # эквивалент "ob-"
    LineStyle(color="orange", marker="D", alpha=0.9),
]

fig, axes = plt.subplots(5, 1, figsize=(10, 15), layout="constrained")
data = [yoga, crossfit, pool, yoga, crossfit]
labels = ["Йога", "Кроссфит", "Бассейн", "ob- (named)", "Йога(2)"]

for ax, st, y, lbl in zip(axes, styles, data, labels):
    ax.plot(months, y, label=lbl, **st.as_kwargs())
    ax.set_title(lbl)
    ax.legend()

# Демонстрация эквивалентности строки "ob-" и именованных аргументов
fig2, ax2 = plt.subplots(figsize=(8, 4), layout="constrained")
ax2.plot(months, yoga, "ob-", label='строка "ob-"')
ax2.plot(months, [v + 5 for v in yoga], color="b", marker="o",
         linestyle="-", label="named kwargs")
ax2.legend()
ax2.set_title('"ob-"  ==  color="b", marker="o", linestyle="-"')

# Комментарии:
# Строка формата читается как [marker][line][color], например "ob-":
# o — маркер-кружок, b — синий, - — сплошная линия. Кратко, но ограничена
# (нельзя задать alpha, linewidth, markersize одновременно).
# markerfacecolor — заливка маркера, markeredgecolor — его контур;
# различать полезно, чтобы выделить точки (белая заливка + тёмный контур).
# alpha<1 делает наложение нескольких линий читаемым — нижние просвечивают.
# linewidth измеряется в точках (1 pt = 1/72 дюйма), а не в пикселях:
# при dpi=200 линия 2 pt будет шире в пикселях, чем при dpi=100 —
# это позволяет сохранять физическую толщину при печати.
# alpha=0 делает линию невидимой, но она остаётся в legend/hit-testing —
# удобно для "скрытых" серий-заглушек или для отладки.
# Цвет можно задать HEX ("#FF6B00") или RGB-кортежем (1.0, 0.42, 0.0) —
# так проще соответствовать брендбуку клуба.

plt.show()
print()



**Задача 5 (деления, сетка, MultipleLocator).** Администратор строит график числа тренировок по дням и хочет видеть деления раз в неделю по оси X и круглые числа (кратные 25) по оси Y. Загрузите `workouts.csv` с датами и постройте линию. Примените `ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))`, `ax.yaxis.set_major_locator(mticker.MultipleLocator(25))`, `ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())`, включите двухуровневую сетку через `ax.grid(which="major", linestyle="--", alpha=0.5)` и `ax.grid(which="minor", linestyle=":", alpha=0.3)`. Ожидаемый вывод: деления по понедельникам и по 25 тренировок; два уровня сетки. В комментариях объясните: чем `Locator` отличается от `Formatter` и почему это разделение ответственности важно; что делает `AutoMinorLocator`; почему `MultipleLocator(25)` удобнее для счётчиков, чем автоматические деления; как `set_xlim(5, -5)` инвертирует ось и зачем это может понадобиться; как `tick_params(which="minor", length=0)` скрывает сами штрихи, оставляя только сетку; что произойдёт при `set_xlim` с диапазоном вне данных — обрежется ли линия или покажется пустая область.

In [ ]:

import numpy as np
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

# --- генерируем workouts.csv ---
np.random.seed(7)
dates = pd.date_range("2024-01-01", periods=120, freq="D")
workouts = np.random.randint(10, 60, size=120)
pd.DataFrame({"date": dates, "workouts": workouts}).to_csv(
    "workouts.csv", index=False
)

# --- загрузка ---
df_w = pd.read_csv("workouts.csv", parse_dates=["date"], index_col="date")

fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.plot(df_w.index, df_w["workouts"], color="steelblue", linewidth=1.5)

# Локаторы
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
ax.yaxis.set_major_locator(mticker.MultipleLocator(25))
ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())

# Двухуровневая сетка
ax.grid(which="major", linestyle="--", alpha=0.5)
ax.grid(which="minor", linestyle=":",  alpha=0.3)

# Скрыть штрихи минорных тиков, оставив только сетку
ax.tick_params(which="minor", length=0)

fig.autofmt_xdate()

# Комментарии:
# Locator решает ГДЕ ставить тики (координаты), Formatter — КАК их
# подписать (текст). Разделение ответственности позволяет независимо
# менять позиции (MultipleLocator/WeekdayLocator) и оформление
# (DateFormatter/WeightFormatter), не переписывая друг друга.
# AutoMinorLocator ставит минорные тики автоматически: обычно n-1
# промежуточный тик между мажорными (по умолчанию n=5 или 4).
# MultipleLocator(25) удобен для счётчиков: тики всегда кратны 25,
# значения круглые; авто-деления давали бы "некрасивые" числа (17, 34, 51).
# set_xlim(5, -5) инвертирует ось: левый край = 5, правый = -5.
# Полезно для инвертированных шкал (глубина, рейтинги "лучше — меньше").
# tick_params(which="minor", length=0) убирает штрихи у минорных тиков,
# но сетка (grid which="minor") по ним остаётся — чистая фоновая сетка.
# set_xlim вне диапазона данных: линия не обрезается "физически", просто
# за пределами окна она не видна; если окно целиком вне данных — увидим
# пустую область, линия за кадром.

plt.show()
print()



**Задача 6 (легенда из groupby).** Фитнес-клуб сравнивает число посещений по типам занятий (йога, пилатес, кроссфит, бассейн, бокс) за квартал. Загрузите `visits_by_class.csv` с колонками `date`, `class_type`, `visits`, постройте линии циклом `for ct, g in df.groupby("class_type"): ax.plot(g["date"], g["visits"], label=ct)`. Вызовите `ax.legend(loc="upper left", ncol=2, title="Типы занятий", framealpha=0.9)`. Дополнительно вынесите легенду за график через `bbox_to_anchor=(1.02, 0.5)`. Ожидаемый вывод: легенда с N записями в 2 колонки; при выносе — за пределами области осей. В комментариях объясните: как `groupby` в Pandas даёт готовые группы для цикла и почему это стандартный паттерн; зачем параметр `ncol=2` и когда он критичен (много типов — 6–8); как `bbox_to_anchor` задаёт точку привязки в долях осей и почему `loc` и `bbox_to_anchor` работают вместе; что делает `borderaxespad=0` и когда его нужно указывать; почему без `label=ct` легенда будет пустой; что произойдёт, если одна из линий не имеет `label` — попадёт ли она в легенду или будет проигнорирована (Matplotlib выдаст предупреждение `No artists with labels found`).


In [ ]:
# --- генерируем visits_by_class.csv ---
np.random.seed(11)
classes = ["йога", "пилатес", "кроссфит", "бассейн", "бокс"]
dates = pd.date_range("2024-01-01", periods=90, freq="D")
rows = []
base = {"йога": 80, "пилатес": 60, "кроссфит": 100, "бассейн": 70, "бокс": 50}
for ct in classes:
    vals = base[ct] + np.cumsum(np.random.randint(-3, 4, size=90))
    for d, v in zip(dates, vals):
        rows.append({"date": d, "class_type": ct, "visits": max(v, 0)})
pd.DataFrame(rows).to_csv("visits_by_class.csv", index=False)

# --- загрузка ---
dfc = pd.read_csv("visits_by_class.csv", parse_dates=["date"])

# --- вариант 1: легенда внутри, 2 колонки ---
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
for ct, g in dfc.groupby("class_type"):
    ax.plot(g["date"], g["visits"], label=ct)
ax.set_title("Посещения по типам занятий (квартал)")
ax.legend(loc="upper left", ncol=2, title="Типы занятий", framealpha=0.9)
fig.savefig("classes_legend_inside.png", dpi=150, bbox_inches="tight")

# --- вариант 2: легенда вынесена вправо ---
fig2, ax2 = plt.subplots(figsize=(10, 6), layout="constrained")
for ct, g in dfc.groupby("class_type"):
    ax2.plot(g["date"], g["visits"], label=ct)
ax2.set_title("Посещения по типам занятий (легенда снаружи)")
ax2.legend(
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    title="Типы занятий",
    borderaxespad=0,
)
fig2.savefig("classes_legend_outside.png", dpi=150, bbox_inches="tight")

# Комментарии:
# df.groupby("class_type") даёт итератор (key, subframe) — готовые группы
# для цикла. Это стандартный паттерн: не нужно фильтровать вручную,
# Pandas сам разбивает по уникальным значениям; порядок — по ключу.
# ncol=2 размещает записи легенды в 2 колонки — критично при 6–8 типах,
# иначе легенда становится длинной и наезжает на данные.
# bbox_to_anchor задаёт точку привязки в системе координат осей (по
# умолчанию в долях ax, x от 0 до 1). loc говорит, КАКАЯ точка легенды
# совпадает с bbox_to_anchor: (1.02, 0.5) + loc="center left" → левый
# центр легенды на 2% правее правого края осей — легенда снаружи.
# borderaxespad=0 убирает доп. отступ между осями и легендой, чтобы
# зазор контролировался только bbox_to_anchor.
# Без label=ct легенда будет пустой — plot не создаёт "подписанного"
# художника, и legend() нечего отображать.
# Если одна из линий без label, она в легенду НЕ попадёт; Matplotlib
# выведет предупреждение "No artists with labels found" только если
# ни у одной линии нет label.

plt.show()
print()


**Задача 7 (bar и barh из value_counts).** Фитнес-клуб анализирует топ-10 самых популярных тренеров по числу проведённых занятий за год. Загрузите `sessions.csv`, получите `top = df["trainer"].value_counts().head(10)`, постройте вертикальную диаграмму `ax.bar(top.index, top.values)` и горизонтальную `ax.barh(top.index[::-1], top.values[::-1])`. Подпишите значения над столбцами через `for i, v in enumerate(top.values): ax.text(i, v + 5, str(v), ha="center")` (для bar) и `ax.text(v + 5, i, str(v), va="center")` (для barh). Ожидаемый вывод: две диаграммы — вертикальная и горизонтальная с перевёрнутыми данными. В комментариях объясните: почему для `barh` данные переворачивают через `[::-1]` и что произойдёт без разворота (самый популярный тренер окажется внизу); почему ось значений bar-графика **должна** начинаться с нуля и что произойдёт при `ax.set_ylim(50, 200)` (визуальный обман); когда `barh` удобнее `bar` (длинные ФИО тренеров типа «Константинопольский Александр»); как `value_counts()` уже сортирует данные по убыванию и зачем `.head(10)`; что произойдёт при `value_counts(normalize=True)` и как это использовать для доли тренера в общем числе занятий; как подписать значения внутри столбцов, а не снаружи (для очень длинных столбцов — иначе текст вылезет за границы).

In [ ]:
# --- генерируем sessions.csv ---
np.random.seed(21)
trainers = [
    "Иванов Пётр", "Смирнова Анна", "Кузнецов Олег", "Попова Мария",
    "Соколов Дмитрий", "Лебедева Ольга", "Козлов Артём", "Новикова Елена",
    "Морозов Игорь", "Волкова Дарья", "Зайцев Никита", "Егорова Светлана",
]
dates = pd.date_range("2024-01-01", periods=500, freq="D")
rows = [{"date": d, "trainer": np.random.choice(trainers, p=np.linspace(.15,.02,12)/sum(np.linspace(.15,.02,12)))} for d in dates]
pd.DataFrame(rows).to_csv("sessions.csv", index=False)

# --- загрузка и топ-10 ---
dfs = pd.read_csv("sessions.csv", parse_dates=["date"])
top = dfs["trainer"].value_counts().head(10)

# --- вертикальная диаграмма ---
fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.bar(top.index, top.values, color="steelblue")
for i, v in enumerate(top.values):
    ax.text(i, v + 2, str(v), ha="center", va="bottom", fontsize=9)
ax.set_title("Топ-10 тренеров по числу занятий (bar)")
ax.set_ylabel("Число занятий")
ax.set_ylim(0, top.values.max() * 1.15)
ax.tick_params(axis="x", rotation=45)

# --- горизонтальная диаграмма ---
fig2, ax2 = plt.subplots(figsize=(10, 6), layout="constrained")
ax2.barh(top.index[::-1], top.values[::-1], color="seagreen")
for i, v in enumerate(top.values[::-1]):
    ax2.text(v + 1, i, str(v), va="center", ha="left", fontsize=9)
ax2.set_title("Топ-10 тренеров по числу занятий (barh)")
ax2.set_xlabel("Число занятий")
ax2.set_xlim(0, top.values.max() * 1.15)

# Доля тренера (normalize)
share = dfs["trainer"].value_counts(normalize=True).head(3)
print("Доля топ-3:", {k: round(v, 3) for k, v in share.items()})

# Комментарии:
# Для barh данные переворачивают [::-1], потому что в matplotlib барх
# растёт СНИЗУ вверх: первый элемент — нижний. Без разворота самый
# популярный тренер окажется внизу, а не сверху — нелогично для "топ".
# Ось значений bar-графика ДОЛЖНА начинаться с нуля: длина столбца
# кодирует величину. set_ylim(50, 200) визуально "растягивает" разницу
# в 2 раза (например, 150 vs 200) и создаёт ложное впечатление обмана.
# barh удобнее bar, когда категории — длинные строки (ФИО тренеров):
# подписи читаются горизонтально, не надо поворачивать на 45°.
# value_counts() уже возвращает Series, отсортированный по убыванию,
# поэтому .head(10) даёт именно ТОП-10, без доп. sort_values().
# value_counts(normalize=True) даёт доли (сумма = 1) — удобно для
# "какая доля занятий у тренера X из всех". Умножение на 100 — проценты.
# Подписи ВНУТРИ столбцов: если bar — ax.text(i, v/2, ...), если barh —
# ax.text(v/2, i, ...). Полезно для столбцов, у которых верх уходит
# за границу оси (иначе текст вылезет за пределы ax).

plt.show()
print()



**Задача 8 (scatter с ручным hue).** Фитнес-клуб изучает связь возраста клиента и максимального веса в становой тяге, с разбивкой по полу (мужчины, женщины). В Matplotlib нет встроенного `hue`, но его легко эмулировать циклом `for gender, g in df.groupby("gender"): ax.scatter(g["age"], g["deadlift_kg"], label=gender, s=50, alpha=0.7, edgecolors="black")`. Ожидаемый вывод: два облака точек разного цвета с общей легендой. В комментариях объясните: чем `scatter` отличается от `plot` при маркерах и почему для диаграммы рассеяния предпочтителен первый; что означает параметр `s` (площадь в точках², а не диаметр) и как увеличить диаметр в 2 раза (площадь в 4 раза); зачем `alpha=0.6` при большом количестве точек; почему `edgecolors="black"` улучшает читаемость на светлом фоне и когда лучше использовать белый; как `groupby` в Pandas даёт готовые срезы для каждой группы; что произойдёт при пропусках в `age` или `deadlift_kg` — нужно ли `dropna()` перед `scatter` и почему (`ValueError: x and y must be the same size`).

In [ ]:
# --- генерируем clients_deadlift.csv ---
np.random.seed(33)
n = 300
age = np.random.randint(18, 65, size=n).astype(float)
gender = np.random.choice(["мужчины", "женщины"], size=n, p=[0.6, 0.4])
base = np.where(gender == "мужчины", 120, 70)
deadlift = base + 0.8 * age + np.random.normal(0, 15, size=n)
deadlift = np.clip(deadlift, 20, None)
dfd = pd.DataFrame({"age": age, "gender": gender, "deadlift_kg": deadlift})
# немного пропусков для демонстрации
dfd.loc[dfd.sample(5, random_state=1).index, "deadlift_kg"] = np.nan
dfd.to_csv("clients_deadlift.csv", index=False)

# --- загрузка ---
dfdl = pd.read_csv("clients_deadlift.csv")

fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
for gender_name, g in dfdl.groupby("gender"):
    g = g.dropna(subset=["age", "deadlift_kg"])
    ax.scatter(
        g["age"], g["deadlift_kg"],
        label=gender_name, s=50, alpha=0.7, edgecolors="black",
    )
ax.set_xlabel("Возраст, лет")
ax.set_ylabel("Становая тяга, кг")
ax.set_title("Связь возраста и становой тяги по полу")
ax.legend(title="Пол")
ax.grid(alpha=0.3)

# Комментарии:
# scatter отличается от plot: plot оптимизирован для ЛИНИЙ (соединяет
# точки по порядку), scatter — для облака точек (каждая точка независима,
# можно задавать массивы цветов/размеров). Для диаграммы рассеяния
# scatter предпочтителен: разные size/color по точкам, отсутствие
# нежелательного "соединения" при несортированных данных.
# s — ПЛОЩАДЬ маркера в точках². Чтобы диаметр вырос в 2 раза,
# площадь надо увеличить в 4 раза: s = 50 → s = 200.
# alpha=0.6–0.7 при большом числе точек уменьшает перекрытие и
# позволяет видеть плотность облака.
# edgecolors="black" даёт контур точке — точки не "сливаются" на
# светлом фоне; белый edge лучше на тёмном фоне / цветных маркерах.
# groupby даёт готовые срезы (key, subframe) — цикл раскрашивает
# каждую группу своим цветом (ручной hue).
# Пропуски в age/deadlift_kg приведут к разной длине x и y —
# ValueError: x and y must be the same size. Поэтому dropna()
# по обеим колонкам перед scatter обязателен.

plt.show()
print()



**Задача 9 (гистограмма с правилом Фридмана-Диакониса).** Фитнес-клуб анализирует распределение длительности тренировок в минутах. Реализуйте функцию `fd_bins(x: pd.Series) -> int`, вычисляющую число корзин по правилу Фридмана-Диакониса: `q25, q75 = np.percentile(x.dropna(), [25, 75])`, `iqr = q75 - q25`, `width = 2 * iqr / len(x) ** (1/3)`, `int((x.max() - x.min()) / width)`. Постройте четыре гистограммы `duration_min`: с `bins=fd_bins(...)`, `bins=10`, `bins=30`, `bins="auto"`. Ожидаемый вывод: FD-вариант обычно даёт 20–30 корзин и балансирует детализацию и гладкость. В комментариях объясните: почему FD-правило устойчиво к выбросам (использует IQR, а не стандартное отклонение); почему мало корзин «сглаживает» пики, а много создаёт «пилу» случайных флуктуаций; зачем `density=True` при сравнении выборок разного размера (например, «новички» vs «опытные» — разное число клиентов); чем `hist` отличается от `bar` (непрерывные vs категориальные данные) и почему для типов занятий нужен `bar`; что произойдёт при `bins=1000` на 500 тренировках (почти все корзины пусты — гистограмма нечитаема); как наложить KDE-кривую через `scipy.stats.gaussian_kde`, чтобы сгладить гистограмму для презентации.

In [ ]:
# --- генерируем durations.csv ---
np.random.seed(45)
n = 500
duration = np.concatenate([
    np.random.normal(45, 8, size=int(n * 0.5)),
    np.random.normal(70, 12, size=int(n * 0.35)),
    np.random.normal(95, 6, size=int(n * 0.15)),
])
duration = np.clip(duration, 15, 130)
pd.DataFrame({"duration_min": duration}).to_csv("durations.csv", index=False)

# --- загрузка ---
dfdur = pd.read_csv("durations.csv")
x = dfdur["duration_min"]

def fd_bins(x: pd.Series) -> int:
    xc = x.dropna()
    q25, q75 = np.percentile(xc, [25, 75])
    iqr = q75 - q25
    if iqr == 0:
        return 10
    width = 2 * iqr / len(xc) ** (1 / 3)
    return max(1, int((xc.max() - xc.min()) / width))

n_fd = fd_bins(x)
print("FD bins:", n_fd)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), layout="constrained")
configs = [
    (axes[0, 0], n_fd,   f"bins=fd_bins ({n_fd})"),
    (axes[0, 1], 10,     "bins=10"),
    (axes[1, 0], 30,     "bins=30"),
    (axes[1, 1], "auto", 'bins="auto"'),
]
for ax, b, title in configs:
    ax.hist(x.dropna(), bins=b, color="steelblue",
            edgecolor="black", alpha=0.75, density=False)
    ax.set_title(title)
    ax.set_xlabel("Длительность, мин")
    ax.set_ylabel("Кол-во тренировок")

# --- KDE-наложение на FD-гистограмму ---
from scipy.stats import gaussian_kde

fig2, ax = plt.subplots(figsize=(10, 6), layout="constrained")
ax.hist(x.dropna(), bins=n_fd, density=True, color="steelblue",
        edgecolor="black", alpha=0.6, label="histogram (density=True)")
kde = gaussian_kde(x.dropna())
xs = np.linspace(x.min(), x.max(), 400)
ax.plot(xs, kde(xs), color="crimson", linewidth=2, label="KDE")
ax.set_title("Гистограмма + KDE (FD-bins)")
ax.set_xlabel("Длительность, мин")
ax.set_ylabel("Плотность")
ax.legend()

# --- демонстрация bins=1000 ---
fig3, ax3 = plt.subplots(figsize=(10, 4), layout="constrained")
ax3.hist(x.dropna(), bins=1000, color="steelblue")
ax3.set_title("bins=1000 на 500 тренировках — почти все корзины пусты")

# Комментарии:
# FD-правило устойчиво к выбросам: использует IQR (25–75 перцентили),
# а не std, поэтому редкие "марафонские" тренировки на 5 часов не
# раздувают ширину корзины.
# Мало корзин (bins=10) сглаживает пики — теряем бимодальность
# (новички 45 мин vs опытные 95 мин). Много корзин (bins=100+) создаёт
# "пилу" случайных флуктуаций — шум вместо структуры.
# density=True нормирует так, что площадь = 1: можно сравнивать
# распределения разного размера ("новички" 100 vs "опытные" 400),
# иначе масштаб задаётся абсолютным count.
# hist — для НЕПРЕРЫВНЫХ величин (длительность): столбцы вплотную,
# границы корзин имеют смысл. bar — для КАТЕГОРИАЛЬНЫХ (типы занятий):
# между категориями нет "промежуточных" значений, столбцы разделены.
# bins=1000 на 500 наблюдениях: большинство корзин пусты, на графике
# "лес" из редких столбиков и максимумов 1 — нечитаемо, детализация
# не несёт информации.
# KDE (scipy.stats.gaussian_kde) рисует гладкую кривую плотности —
# удобно для презентации: видна бимодальность без "пилы" гистограммы;
# рисуется с density=True, чтобы масштаб совпадал.

plt.show()
print()



**Задача 10 (круговая диаграмма и donut).** Фитнес-клуб визуализирует структуру выручки по источникам (абонементы, персональные тренировки, групповые занятия, магазин спорпита, аренда, прочее). Постройте `ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})`. Второй вариант — donut через `wedgeprops={"width": 0.4}` и текст в центре через `ax.text(0, 0, f"Всего: {counts.sum():,}", ha="center", va="center")`. Ожидаемый вывод: круговая диаграмма с процентами; donut с общей суммой в центре. В комментариях объясните: как `autopct` форматирует проценты и зачем `%%`; как `startangle=90` поворачивает диаграмму и почему это влияет на восприятие (первый сектор начинается сверху — как на часах); зачем `explode` и когда его использовать (выделить нишевый, но важный сегмент — магазин спорпита); почему `pie` плохо работает при >7 категориях и близких значениях; как `wedgeprops={"width": 0.4}` превращает круговую в donut; что произойдёт при отрицательных значениях (ошибка `ValueError: Wedge sizes must be non-negative`) и как обрабатывать возвраты абонементов (переводить в положительные перед `pie`).


In [ ]:
# --- данные о выручке ---
counts = pd.Series({
    "Абонементы": 4_200_000,
    "Персональные": 1_800_000,
    "Групповые": 1_100_000,
    "Спортпит": 450_000,
    "Аренда": 300_000,
    "Прочее": 150_000,
})

# --- круговая ---
fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
ax.pie(
    counts.values,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
ax.set_title("Структура выручки фитнес-клуба")

# --- donut с выделением "Спортпит" ---
fig2, ax2 = plt.subplots(figsize=(8, 8), layout="constrained")
explode = [0, 0, 0, 0.12, 0, 0]  # выделяем "Спортпит"
ax2.pie(
    counts.values,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=90,
    explode=explode,
    wedgeprops={"width": 0.4, "edgecolor": "white", "linewidth": 2},
)
ax2.text(
    0, 0, f"Всего: {counts.sum():,}",
    ha="center", va="center", fontsize=12, fontweight="bold",
)
ax2.set_title("Donut: выручка по источникам")

# Комментарии:
# autopct="%1.1f%%" — строка формата printf: %1.1f — число с 1 знаком
# после точки, %% — экранированный знак процента (иначе Python примет
# один "%" за начало спецификатора). Результат: "52.5%".
# startangle=90 поворачивает: первый сектор начинается сверху (12 часов),
# далее по часовой стрелке — привычная "циферблатная" логика, легче
# читается, чем старт с 3 часов (0°).
# explode выделяет сегмент (сдвигает от центра). Уместно для нишевого,
# но важного: "Спортпит" — маленький, но растущий, его визуально
# подсвечиваем, чтобы привлечь внимание управляющей компании.
# pie плохо работает при >7 категориях и близких значениях: углы
# неразличимы глазом, подписи наезжают, проценты теряют смысл.
# Для >7 категорий лучше barh или "топ-5 + прочее".
# wedgeprops={"width": 0.4} — задаёт толщину кольца (40% радиуса),
# остальное "вырезается" — получается donut. Центр свободен под текст
# (общая сумма, средний чек, KPI).
# Отрицательные значения дают ValueError: Wedge sizes must be
# non-negative. Возвраты абонементов нужно либо вычитать из
# соответствующей категории до pie (получить положительный итог),
# либо показывать отдельной положительной категорией "Возвраты",
# либо нормировать (abs / клиппинг).

plt.show()
print()


**Задача 11 (экспорт в несколько форматов и BytesIO).** Фитнес-клуб готовит отчёт для управляющей компании: графики нужны в PNG (для сайта), PDF (для печати), SVG (для редактирования), JPEG (для email-рассылки) и прозрачный PNG (для слайда с фирменным фоном). Постройте график и сохраните в 5 форматов: `report.png` с `dpi=300`, `report.pdf`, `report.svg`, `report.jpg` с `dpi=100`, `report_transparent.png` с `transparent=True`. Для каждого файла вычислите SHA-256 через `hashlib.sha256(open(name, "rb").read()).hexdigest()[:8]` и размер через `os.path.getsize`. Дополнительно сохраните в `BytesIO` (для последующей отправки через FastAPI) и верните длину. Ожидаемый вывод: размеры и хэши 5 файлов; PNG-прозрачный — с альфа-каналом. В комментариях объясните: зачем нужны хэши файлов (детерминизм, кэширование CI, проверка изменений при ревью); почему `savefig` нужно вызывать до `plt.close(fig)`, а не после; как `BytesIO` используется в FastAPI (`Response(content=buf.getvalue(), media_type="image/png")`) и зачем `buf.seek(0)` перед чтением; почему JPEG плох для графиков (артефакты сжатия на резких границах текста); как `metadata={"Title": "Отчёт фитнес-клуба", "Author": "..."}` встраивает информацию в PDF; что произойдёт при сохранении в несуществующую папку (`FileNotFoundError`) и как `Path("outputs").mkdir(parents=True, exist_ok=True)` решает проблему.

**Задача 12 (стили, rcParams, контекстный менеджер).** Фитнес-клуб хочет, чтобы все внутренние графики имели единый стиль, но при этом титульные графики для инвесторов выглядели иначе. Реализуйте `@contextlib.contextmanager def style_ctx(name): with plt.style.context(name): yield`. Примените `with style_ctx("ggplot"):` к одному графику, затем `with style_ctx("seaborn-v0_8-darkgrid"):` к другому. Дополнительно покажите `plt.rcParams.update({"font.size": 12, "axes.grid": True, "grid.alpha": 0.3})` и `mpl.rcdefaults()` для полного сброса. Ожидаемый вывод: графики в разных стилях; после `rcdefaults` — заводской вид. В комментариях объясните: чем `plt.style.use` отличается от `plt.style.context` (глобально vs временно — второе критично в Jupyter); зачем `rcParams` для тонкой настройки; как комбинировать стили (`use(["seaborn-v0_8", "dark_background"])`) — какой перекрывает какой; почему `plt.style.use("default")` **не** сбрасывает вручную заданные `rcParams` и что использовать (`mpl.rcdefaults()`); где хранятся собственные `.mplstyle` файлы (`~/.config/matplotlib/stylelib/`) и как загрузить стиль **по имени**; что вернёт `plt.style.available` и как быстро посмотреть доступные стили для вдохновения.

---



## 🔵 Уровень 2. Ученик (задачи 13–22)

**Задача 13 (аннотации с argmax/argmin).** Фитнес-клуб анализирует дневную посещаемость за месяц и хочет выделить дни-пики и дни-провалы (например, 2 января и «мёртвое» воскресенье летом). Загрузите `daily_visits.csv` с `parse_dates=["date"], index_col="date"`, найдите `idx_max = df["visits"].idxmax()` и `idx_min = df["visits"].idxmin()`. Добавьте аннотации через `ax.annotate(f"Пик: {df.loc[idx_max, 'visits']:,}", xy=(idx_max, df.loc[idx_max, "visits"]), xytext=(10, 20), textcoords="offset points", arrowprops=dict(arrowstyle="->", color="green", connectionstyle="arc3,rad=0.2"))`; аналогично для минимума с `xytext=(-80, -30)`, `color="red"`. Ожидаемый вывод: стрелки указывают на пики, текст смещён и не перекрывает линию. В комментариях объясните: чем `textcoords="offset points"` отличается от `"data"` и почему первый удобнее для смещения текста; почему `idxmax()` в Pandas возвращает **метку индекса**, а не позицию, и как это используется в `xy`; зачем `arc3,rad=0.2` (изогнутая стрелка) и как `rad=-0.2` развернёт изгиб; как `bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", edgecolor="orange")` в `annotate` улучшает читаемость поверх линий; что произойдёт без `ha`/`va` (текст «прилипнет» к точке и перекроет маркер); как аннотировать несколько близких точек, чтобы они не перекрывались.

In [ ]:
# --- генерируем daily_visits.csv ---
np.random.seed(55)
dates = pd.date_range("2024-01-01", periods=60, freq="D")
base = 150 + 40 * np.sin(np.arange(60) / 6)
visits = base + np.random.normal(0, 15, size=60)
visits[1] = 320   # пик — 2 января
visits[10] = 40   # провал
visits = np.clip(visits, 20, None).astype(int)
pd.DataFrame({"date": dates, "visits": visits}).to_csv(
    "daily_visits.csv", index=False
)

# --- загрузка ---
dfv = pd.read_csv("daily_visits.csv", parse_dates=["date"], index_col="date")
idx_max = dfv["visits"].idxmax()
idx_min = dfv["visits"].idxmin()

fig, ax = plt.subplots(figsize=(12, 6), layout="constrained")
ax.plot(dfv.index, dfv["visits"], color="steelblue", linewidth=1.5)
ax.scatter([idx_max, idx_min], [dfv.loc[idx_max, "visits"],
                                dfv.loc[idx_min, "visits"]],
           color=["green", "red"], s=60, zorder=5)

ax.annotate(
    f"Пик: {dfv.loc[idx_max, 'visits']:,}",
    xy=(idx_max, dfv.loc[idx_max, "visits"]),
    xytext=(10, 20),
    textcoords="offset points",
    ha="left", va="bottom",
    bbox=dict(boxstyle="round,pad=0.5",
              facecolor="lightyellow", edgecolor="orange"),
    arrowprops=dict(arrowstyle="->", color="green",
                    connectionstyle="arc3,rad=0.2"),
)

ax.annotate(
    f"Провал: {dfv.loc[idx_min, 'visits']:,}",
    xy=(idx_min, dfv.loc[idx_min, "visits"]),
    xytext=(-80, -30),
    textcoords="offset points",
    ha="right", va="top",
    bbox=dict(boxstyle="round,pad=0.5",
              facecolor="#ffe6e6", edgecolor="red"),
    arrowprops=dict(arrowstyle="->", color="red",
                    connectionstyle="arc3,rad=0.2"),
)

ax.set_title("Дневная посещаемость: пик и провал")
ax.set_ylabel("Посещения")

# Комментарии:
# textcoords="offset points" — смещение xytext в ТОЧКАХ (pt) от xy,
# не зависит от масштаба данных: при изменении осей смещение остаётся
# визуально тем же. "data" трактует xytext как координаты данных —
# текст "уедет" при изменении диапазона.
# idxmax() возвращает МЕТКУ индекса (Timestamp), а не позицию (int).
# В xy=(idx_max, ...) это идеально — ax знает дату по оси X. Для
# позиции использовали бы argmax() + .iloc.
# arc3,rad=0.2 — изогнутая стрелка (радиус кривизны 20% длины).
# rad=-0.2 развернёт изгиб в другую сторону — удобно, чтобы стрелки
# от близких точек расходились, а не пересекались.
# bbox с boxstyle="round,pad=0.5" — плашка под текстом: читается
# поверх линий/сетки; facecolor/edgecolor — светлый фон + цвет рамки.
# Без ha/va текст "прилипнет" к xy-точке и перекроет маркер; ha/va
# явно выравнивают текст относительно offset-точки.
# Несколько близких точек: чередуют знаки смещения (10,20) и (-80,-30),
# разные rad у стрелок, иногда — ручной подбор xytext под каждую точку;
# иначе подписи сольются.

plt.show()
print()



**Задача 14 (продвинутая легенда из groupby).** Фитнес-клуб имеет 8 тренеров и хочет показать на графике только 2 ключевых (например, «Иванов» и «Смирнова»), но остальные оставить построенными — чтобы видеть общий фон. Загрузите `sessions_by_trainer.csv`, постройте линии циклом `for tr, g in df.groupby("trainer"): line, = ax.plot(...); lines[tr] = line`. Создайте две легенды: выборочную через `ax.legend(handles=[lines["Иванов"], lines["Смирнова"]], loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0)` — вынесенную за график, и полную через `ncol=2` внутри. Ожидаемый вывод: две легенды — выборочная вне графика и полная внутри. В комментариях объясните: зачем запятая в `line, = ax.plot(...)` (распаковка кортежа из одного элемента); что произойдёт без запятой и почему `handles=[line]` не сработает (`AttributeError: 'tuple' object has no attribute 'set_label'`); как `bbox_to_anchor` в долях осей задаёт точку привязки и почему `loc="center left"` + `bbox_to_anchor=(1.02, 0.5)` помещает легенду справа; что делает `borderaxespad=0`; почему при многих категориях **полная** легенда перегружает график и лучше использовать выборочную; как автоматически отсортировать категории через `sorted(lines.items())`, чтобы в легенде был алфавитный порядок.

In [ ]:
# --- генерируем sessions_by_trainer.csv ---
np.random.seed(66)
trainers8 = ["Иванов", "Смирнова", "Кузнецов", "Попова",
             "Соколов", "Лебедева", "Козлов", "Новикова"]
months = pd.date_range("2024-01-01", periods=12, freq="MS")
rows = []
for i, tr in enumerate(trainers8):
    base = 20 + i * 3
    vals = base + np.cumsum(np.random.randint(-2, 4, size=12))
    for m, v in zip(months, vals):
        rows.append({"month": m, "trainer": tr, "sessions": max(v, 0)})
pd.DataFrame(rows).to_csv("sessions_by_trainer.csv", index=False)

# --- загрузка ---
dft = pd.read_csv("sessions_by_trainer.csv", parse_dates=["month"])

# --- вариант 1: выборочная легенда снаружи ---
fig, ax = plt.subplots(figsize=(11, 6), layout="constrained")
lines = {}
for tr, g in dft.groupby("trainer"):
    line, = ax.plot(g["month"], g["sessions"], label=tr, marker="o",
                    markersize=4)
    lines[tr] = line

# Перекрасим неключевых в серый, ключевых — в цвета
for tr, ln in lines.items():
    if tr not in ("Иванов", "Смирнова"):
        ln.set_color("lightgray")
        ln.set_linewidth(1)
ax.set_title("Ключевые тренеры на фоне остальных")

ax.legend(
    handles=[lines["Иванов"], lines["Смирнова"]],
    labels=["Иванов", "Смирнова"],
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    borderaxespad=0,
    title="Ключевые",
)
fig.savefig("trainers_selected_legend.png", dpi=150, bbox_inches="tight")

# --- вариант 2: полная легенда внутри, ncol=2 ---
fig2, ax2 = plt.subplots(figsize=(11, 6), layout="constrained")
lines2 = {}
for tr, g in dft.groupby("trainer"):
    line, = ax2.plot(g["month"], g["sessions"], label=tr, marker="o",
                     markersize=4)
    lines2[tr] = line
ax2.set_title("Полная легенда (8 тренеров)")
ax2.legend(loc="upper left", ncol=2, fontsize=8, framealpha=0.9)
fig2.savefig("trainers_full_legend.png", dpi=150, bbox_inches="tight")

# --- алфавитный порядок в легенде ---
fig3, ax3 = plt.subplots(figsize=(11, 6), layout="constrained")
lines3 = {}
for tr, g in dft.groupby("trainer"):
    line, = ax3.plot(g["month"], g["sessions"], label=tr)
    lines3[tr] = line
ax3.legend(
    handles=[ln for _, ln in sorted(lines3.items())],
    labels=[k for k, _ in sorted(lines3.items())],
    loc="upper left", ncol=2, fontsize=8,
)
ax3.set_title("Легенда в алфавитном порядке")

# Комментарии:
# Запятая в "line, = ax.plot(...)" — распаковка кортежа из одного
# элемента: plot возвращает [Line2D], запятая "разворачивает" его в line.
# Без запятой line = [Line2D] (список) — handles=[line] положит в
# legend список, а не Line2D; Matplotlib попытается вызвать
# line.set_label() на list → AttributeError: 'tuple' object has no
# attribute 'set_label' (или аналог для list).
# bbox_to_anchor=(1.02, 0.5) в долях осей: x=1.02 — чуть правее
# правого края ax, y=0.5 — середина по вертикали. loc="center left"
# говорит: левый центр легенды совпадает с этой точкой — легенда
# полностью правее графика, по центру.
# borderaxespad=0 убирает стандартный отступ между осями и легендой,
# чтобы зазор задавался только bbox_to_anchor (точный контроль).
# Полная легенда при 8+ категориях перегружает: занимает много места,
# наезжает на данные, цвета трудно сопоставить с линиями. Выборочная
# (2 ключевых) + серый фон остальных = акцентный рассказ.
# sorted(lines.items()) сортирует по ключу (имени тренера), затем
# распаковка в handles и labels даёт алфавитный порядок в легенде —
# предсказуемо и профессионально.

plt.show()
print()



**Задача 15 (группированные столбцы из pivot_table).** Фитнес-клуб сравнивает число новых клиентов по типам абонементов за 2023 и 2024 год. Постройте `pivot = members.pivot_table(index="membership", columns="year", values="new_clients", aggfunc="sum")`, получите `x = np.arange(len(pivot.index))` и `width = 0.8 / len(pivot.columns)`. Через цикл `for i, year in enumerate(pivot.columns): offset = (i - len(pivot.columns) / 2 + 0.5) * width; ax.bar(x + offset, pivot[year], width, label=str(year))` постройте сгруппированные столбцы. Установите `ax.set_xticks(x)` и `ax.set_xticklabels(pivot.index)`. Ожидаемый вывод: парные столбцы (2023 и 2024) для каждого типа абонемента. В комментариях объясните: почему `width = 0.8 / n_years`, а не фиксированный `0.35` (иначе столбцы перекроются при 3+ группах); как формула `(i - n/2 + 0.5) * width` обобщается на любое число групп; как `pivot_table` сразу готовит данные — что было бы, если бы данных не было в правильной форме; зачем `set_xticks` и `set_xticklabels` по отдельности; что произойдёт при `width=0.8` с двумя группами (полностью перекроются); как добавить третью группу (например, прогноз на 2025) и почему это работает без изменений в коде.

In [ ]:
# --- генерируем members.csv ---
np.random.seed(77)
memberships = ["Базовый", "Стандарт", "Премиум", "VIP"]
years = [2023, 2024]
rows = []
for m in memberships:
    for y in years:
        for _ in range(np.random.randint(5, 15)):
            rows.append({
                "membership": m,
                "year": y,
                "new_clients": np.random.randint(10, 100),
            })
pd.DataFrame(rows).to_csv("members.csv", index=False)

# --- загрузка и pivot ---
members = pd.read_csv("members.csv")
pivot = members.pivot_table(
    index="membership", columns="year",
    values="new_clients", aggfunc="sum",
)

x = np.arange(len(pivot.index))
n = len(pivot.columns)
width = 0.8 / n

fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
for i, year in enumerate(pivot.columns):
    offset = (i - n / 2 + 0.5) * width
    ax.bar(x + offset, pivot[year], width, label=str(year))

ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_xlabel("Тип абонемента")
ax.set_ylabel("Новые клиенты")
ax.set_title("Новые клиенты по типам абонементов, 2023 vs 2024")
ax.legend(title="Год")

# --- демонстрация третьей группы (прогноз) ---
pivot3 = pivot.copy()
pivot3[2025] = (pivot3[2023] + pivot3[2024]) * 0.55  # прогноз
pivot3 = pivot3[[2023, 2024, 2025]]

x3 = np.arange(len(pivot3.index))
n3 = len(pivot3.columns)
width3 = 0.8 / n3

fig2, ax2 = plt.subplots(figsize=(10, 6), layout="constrained")
for i, year in enumerate(pivot3.columns):
    offset = (i - n3 / 2 + 0.5) * width3
    ax2.bar(x3 + offset, pivot3[year], width3, label=str(year))
ax2.set_xticks(x3)
ax2.set_xticklabels(pivot3.index)
ax2.set_title("+ прогноз 2025 (код не изменился)")
ax2.legend(title="Год")

# Комментарии:
# width = 0.8 / n_years — общая "ширина слота" 0.8 делится на все
# группы. Фиксированный 0.35 при 3+ группах дал бы перекрытие
# столбцов (3 × 0.35 = 1.05 > 1.0). При 2 группах 0.35 работал бы
# случайно (0.7 < 1.0), но не масштабируется.
# Формула offset = (i - n/2 + 0.5) * width: для n=2, i=0 → -0.5·width,
# i=1 → +0.5·width (симметрично вокруг центра x). Для n=3: -width, 0,
# +width. Обобщается на любое n.
# pivot_table сразу сворачивает данные в "широкую" форму
# (индекс — тип, столбцы — годы), готовую к циклу по годам.
# Без pivot пришлось бы вручную фильтровать по year и строить bar
# с собственными сдвигами — больше кода и риск ошибки.
# set_xticks(x) ставит ПОЗИЦИИ тиков (центры слотов), а
# set_xticklabels(pivot.index) — их ТЕКСТ. Разделение позволяет
# независимо менять подписи (сокращать, переводить) без пересчёта
# координат.
# width=0.8 на 2 группы: 2 × 0.8 = 1.6, столбцы полностью перекроются
# (соседние бары наедут друг на друга и на соседние слоты).
# Третья группа (2025) добавляется без изменений в теле цикла —
# достаточно добавить столбец в pivot и пересчитать width/n. Код уже
# обобщён через n = len(pivot.columns).

plt.show()
print()



**Задача 16 (stacked bar + errorbar).** Фитнес-клуб показывает структуру выручки по источникам (абонементы, персональные, групповые) для каждого филиала. Постройте `pivot = revenue.pivot_table(index="branch", columns="source", values="amount_rub", aggfunc="sum")`, реализуйте накопление через `bottom = np.zeros(len(pivot)); for src in pivot.columns: ax.bar(pivot.index, pivot[src], bottom=bottom, label=src); bottom += pivot[src].values`. Добавьте планки ошибок к **сумме** через `ax.errorbar(pivot.index, bottom, yerr=stds.values, fmt="none", capsize=8, ecolor="black")`. Ожидаемый вывод: накопленные столбцы с планками ошибок наверху каждого. В комментариях объясните: как `bottom=bottom` реализует накопление и почему `bottom += pivot[src].values` (а не `bottom = pivot[src]`) — что будет, если забыть `+=`; почему `yerr` нужен для **суммы**, а не для отдельного источника (планка «утонет» внутри столбца и будет вводить в заблуждение); зачем `fmt="none"` (только планки, без маркеров); как `error_kw={"elinewidth": 2, "ecolor": "black"}` объединяет стиль; что произойдёт при неправильном порядке категорий в `bottom` и `yerr` (планки окажутся не на тех столбцах); как вычислить стандартное отклонение по филиалам (`revenue.groupby("branch")["amount_rub"].std()`).



**Задача 17 (scatter + polyfit + выбросы по z-score).** Фитнес-клуб исследует связь числа тренировок в неделю и процента жира — но в данных есть «аварийные» дни с аномально низким процентом жира при малом числе тренировок (генетика, диета, болезнь). Постройте облако точек через `ax.scatter(df["workouts_per_week"], df["body_fat_pct"], alpha=0.6)`, вычислите `m, b = np.polyfit(df["workouts_per_week"], df["body_fat_pct"], 1)` и нарисуйте тренд `ax.plot(x_line, m * x_line + b, "r--")`. Выбросы определите по `z = np.abs((df["body_fat_pct"] - df["body_fat_pct"].mean()) / df["body_fat_pct"].std()); mask = z > 3` и выделите отдельным `ax.scatter(df.loc[mask, "workouts_per_week"], df.loc[mask, "body_fat_pct"], c="red", marker="X", s=200, edgecolors="black", label="Выбросы", zorder=5)`. Добавьте в заголовок `r = np.corrcoef(df["workouts_per_week"], df["body_fat_pct"])[0, 1]`. В комментариях объясните: что означает `m` в `polyfit` (наклон — на сколько процентов жира снижается каждая тренировка в неделю); почему `polyfit` чувствителен к выбросам и как `RANSACRegressor` из `sklearn.linear_model` решает проблему; как `zorder` держит выбросы поверх точек; что произойдёт при нелинейной зависимости — линейный тренд введёт в заблуждение и нужен `polyfit(deg=2)`; как `np.corrcoef[0, 1]` извлекает коэффициент корреляции Пирсона из матрицы; почему коэффициент Пирсона чувствителен к выбросам, а Спирмен (`scipy.stats.spearmanr`) — нет.


In [ ]:
# --- генерируем body_fat.csv ---
np.random.seed(99)
n = 200
workouts = np.random.uniform(1, 6, size=n)
body_fat = 30 - 2.0 * workouts + np.random.normal(0, 2.5, size=n)
# добавим выбросы: аномально низкий % жира при малом числе тренировок
idx_out = np.random.choice(n, 6, replace=False)
body_fat[idx_out] = np.random.uniform(8, 12, size=6)
body_fat = np.clip(body_fat, 5, 40)
dfbf = pd.DataFrame({"workouts_per_week": workouts, "body_fat_pct": body_fat})
dfbf.to_csv("body_fat.csv", index=False)

# --- загрузка ---
dfb = pd.read_csv("body_fat.csv")

fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")

ax.scatter(dfb["workouts_per_week"], dfb["body_fat_pct"],
           alpha=0.6, color="steelblue", label="Клиенты")

# Линейный тренд
m, b = np.polyfit(dfb["workouts_per_week"], dfb["body_fat_pct"], 1)
x_line = np.linspace(dfb["workouts_per_week"].min(),
                     dfb["workouts_per_week"].max(), 100)
ax.plot(x_line, m * x_line + b, "r--", label=f"тренд: y={m:.2f}x+{b:.2f}")

# Выбросы по z-score
z = np.abs((dfb["body_fat_pct"] - dfb["body_fat_pct"].mean())
           / dfb["body_fat_pct"].std())
mask = z > 3
ax.scatter(
    dfb.loc[mask, "workouts_per_week"], dfb.loc[mask, "body_fat_pct"],
    c="red", marker="X", s=200, edgecolors="black",
    label="Выбросы", zorder=5,
)

# Корреляция Пирсона
r = np.corrcoef(dfb["workouts_per_week"], dfb["body_fat_pct"])[0, 1]
ax.set_title(f"Тренировки vs % жира (r Пирсона = {r:.2f})")
ax.set_xlabel("Тренировок в неделю")
ax.set_ylabel("Процент жира")
ax.legend()
ax.grid(alpha=0.3)

# --- сравнение: RANSAC vs polyfit ---
from sklearn.linear_model import RANSACRegressor

X = dfb[["workouts_per_week"]].values
y = dfb["body_fat_pct"].values
ransac = RANSACRegressor(random_state=0).fit(X, y)
x_r = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_r = ransac.predict(x_r)

fig2, ax2 = plt.subplots(figsize=(10, 6), layout="constrained")
ax2.scatter(X, y, alpha=0.5, color="steelblue", label="данные")
ax2.plot(x_line, m * x_line + b, "r--", label="polyfit (OLS)")
ax2.plot(x_r, y_r, "g-", linewidth=2, label="RANSAC (устойчив к выбросам)")
ax2.legend()
ax2.set_title("Сравнение OLS и RANSAC на данных с выбросами")

# Комментарии:
# m в polyfit(deg=1) — НАКЛОН: на сколько процентов жира в среднем
# меняется % жира при +1 тренировке в неделю (отрицательное — снижение).
# polyfit (МНК) минимизирует сумму квадратов — выбросы с большим
# отклонением "тянут" прямую на себя. RANSAC итеративно подбирает
# модель по случайным подмножествам и отбрасывает точки-выбросы
# (inliers) — тренд получается близким к "чистым" данным.
# zorder=5 держит выбросы ПОВЕРХ облака (zorder обычного scatter ~1-2),
# иначе они могут быть не видны под плотными точками.
# При НЕЛИНЕЙНОЙ зависимости линейный polyfit(deg=1) даёт смещённый
# наклон и вводит в заблуждение — нужен polyfit(deg=2) или KDE/spline.
# np.corrcoef возвращает матрицу 2×2; [0, 1] берёт вне-диагональный
# элемент — сам коэффициент Пирсона r ∈ [-1, 1].
# Пирсон чувствителен к выбросам (использует среднее/std), Спирмен
# (scipy.stats.spearmanr) работает с РАНГАМИ — выброс остаётся
# крайним рангом и не раздувает коэффициент.

plt.show()
print()


**Задача 18 (density, cumulative, step).** Фитнес-клуб сравнивает распределения длительности тренировок по трём типам (йога, кроссфит, бассейн). Если построить обычные гистограммы, тип с малым числом клиентов будет визуально «раздавлен» массовым. Постройте 4 подграфика: (а) наложение с `alpha=0.5`; (б) `density=True` (нормировка площади к 1); (в) `cumulative=True` (накопленная частота); (г) `histtype="step"` (только контуры). Для каждого `for ct, g in df.groupby("class_type"): ax.hist(g["duration_min"], ...)`. В комментариях объясните: зачем `density=True` при разном `n`; как `cumulative=True` находит медиану (точка, где накопленная частота = 0.5) и как её отметить через `ax.axvline(median_value)`; почему `histtype="step"` лучше при 3+ сериях; как зафиксировать `bins` при сравнении (`bins=np.linspace(min, max, 21)`) — что будет при автоматическом подборе корзин для каждой группы отдельно; как наложить KDE через `scipy.stats.gaussian_kde`; что произойдёт при `cumulative=-1` (обратная накопленная — от 1 к 0, полезно для «доли тренировок короче X»).


In [ ]:
# --- генерируем durations_by_class.csv ---
np.random.seed(111)
rows = []
specs = {
    "йога":     (60, 8,  400),
    "кроссфит": (50, 10, 120),
    "бассейн":  (45, 6,  80),
}
for ct, (mu, sd, n) in specs.items():
    dur = np.clip(np.random.normal(mu, sd, size=n), 15, 120)
    for d in dur:
        rows.append({"class_type": ct, "duration_min": d})
pd.DataFrame(rows).to_csv("durations_by_class.csv", index=False)

# --- загрузка ---
dfd2 = pd.read_csv("durations_by_class.csv")

# общие корзины для всех групп
bins = np.linspace(dfd2["duration_min"].min(),
                   dfd2["duration_min"].max(), 21)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")
titles = [
    "наложение (alpha=0.5)",
    "density=True (площадь = 1)",
    "cumulative=True (накопленная)",
    'histtype="step" (контуры)',
]
kwargs_list = [
    dict(alpha=0.5),
    dict(density=True, alpha=0.5),
    dict(cumulative=True, density=True, alpha=0.5),
    dict(histtype="step", linewidth=2),
]

for ax, title, kw in zip(axes.flat, titles, kwargs_list):
    for ct, g in dfd2.groupby("class_type"):
        ax.hist(g["duration_min"], bins=bins, label=ct, **kw)
    ax.set_title(title)
    ax.set_xlabel("Длительность, мин")
    ax.set_ylabel("Плотность" if "density" in kw or "cumulative" in kw
                  else "Частота")
    ax.legend()

# --- медиана на cumulative-графике ---
ax_cum = axes[1, 0]
for ct, g in dfd2.groupby("class_type"):
    med = g["duration_min"].median()
    ax_cum.axvline(med, linestyle="--", alpha=0.7)

# --- KDE-наложение на step-график ---
from scipy.stats import gaussian_kde

ax_step = axes[1, 1]
xs = np.linspace(bins[0], bins[-1], 400)
for ct, g in dfd2.groupby("class_type"):
    kde = gaussian_kde(g["duration_min"])
    ax_step.plot(xs, kde(xs), linewidth=2, label=f"{ct} (KDE)")
ax_step.legend(fontsize=8)

# --- демонстрация cumulative=-1 ---
fig2, ax2 = plt.subplots(figsize=(10, 5), layout="constrained")
for ct, g in dfd2.groupby("class_type"):
    ax2.hist(g["duration_min"], bins=bins, cumulative=-1,
             density=True, histtype="step", linewidth=2, label=ct)
ax2.set_title("cumulative=-1: доля тренировок ДЛИННЕЕ X")
ax2.set_xlabel("Длительность, мин")
ax2.set_ylabel("Доля (1 → 0)")
ax2.legend()

# Комментарии:
# density=True нормирует гистограмму так, что площадь = 1. Это
# необходимо при разном n: иначе массовая "йога" (400) визуально
# задавит "бассейн" (80) по высоте столбцов, хотя формы распределений
# сопоставимы. density позволяет сравнивать ФОРМУ, а не абсолютные count.
# cumulative=True — накопленная частота: значение в точке x = доля
# наблюдений ≤ x. Медиана — там, где кривая пересекает 0.5 по Y;
# отмечаем через axvline(median) вертикальной линией.
# histtype="step" рисует только контуры (без заливки) — при 3+ сериях
# наложение заливок даёт "грязь", контуры же читаются поверх друг друга.
# Фиксированные bins=np.linspace(min, max, 21) важны при сравнении:
# при автоматическом подборе корзин КАЖДАЯ группа получит свои границы,
# гистограммы будут несопоставимы по X (пики "съедут").
# KDE (gaussian_kde) — гладкая кривая плотности; накладывается через
# plot, нормализуется автоматически (сравнивать с density=True).
# cumulative=-1 — обратная накопленная: 1 в левом краю → 0 в правом.
# Значение в x = доля тренировок ДЛИННЕЕ x. Удобно отвечать на вопрос
# "какая доля тренировок длится дольше 60 минут?".

plt.show()
print()


**Задача 19 (stackplot и fill_between для диапазонов).** Фитнес-клуб показывает структуру выручки по источникам (абонементы, персональные, групповые) и отдельно — доверительный интервал для прогноза онлайн-продаж абонементов. Постройте `ax.stackplot(dates, memberships, personal, group, labels=["Абонементы", "Персональные", "Групповые"], alpha=0.8)` — суммируемые данные. Для доверительного интервала прогноза постройте `lo = forecast.rolling(7).quantile(0.1); hi = forecast.rolling(7).quantile(0.9); ax.fill_between(dates, lo, hi, alpha=0.2, color="steelblue")`. Ожидаемый вывод: накопленные области и CI-полоса на второй оси или на отдельном графике. В комментариях объясните: когда `stackplot` (суммируемые данные), когда `fill_between` (диапазоны — CI, оптимистичный/пессимистичный прогноз); что делает `baseline="sym"` (симметрично относительно нуля — полезно для знакопеременных данных типа прибыли/убытка); как `where=` закрашивает только часть (`fill_between(..., where=(y > threshold))`); что произойдёт с `stackplot` при 8+ категориях и как объединить мелкие; как `stackplot` связан с накопленным `bar(bottom=...)` и когда что использовать; как аннотировать проценты подписей внутри каждой области.

**Задача 20 (pie топ-N с «Прочее»).** Фитнес-клуб визуализирует распределение клиентов по целям тренировок — их десятки, но в презентации нужны только топ-5 плюс «Прочее». Получите `counts = df["goal"].value_counts()`, оставьте `top5 = counts.head(5)`, посчитайте `other = counts.iloc[5:].sum()`, объедините через `pd.concat([top5, pd.Series({"Прочее": other})])`. Постройте `ax.pie(combined.values, labels=combined.index, autopct="%1.1f%%", explode=(0.05, 0, 0, 0, 0, 0), wedgeprops={"edgecolor": "white", "linewidth": 2})`. Ожидаемый вывод: 6 секторов, где «Прочее» визуально меньше остальных, но вклад виден по подписи. В комментариях объясните: как `pd.concat` объединяет Series и почему `pd.Series({"Прочее": other})` создаёт Series с одним элементом; зачем группировать мелкие категории (читаемость — 20 мелких секторов сливаются в кашу); когда `pie` вообще уместна (доли целого, ≤7 категорий, положительные суммируемые данные); как `autopct=lambda p: f"{p * sum / 100:.0f}"` показывает абсолютные значения вместо процентов; как отсортировать категории по размеру перед `pie` — что важнее, размер или алфавитный порядок; почему `explode=(0.05, 0, 0, ...)` выделяет первый сектор.

**Задача 21 (twinx на данных фитнес-клуба).** Фитнес-клуб визуализирует число посещений и среднюю выручку на клиента (ARPU) — две переменные с разными масштабами (посещения 50–500 в день, ARPU 1500–5000 ₽). Постройте число посещений на `ax1` столбцами и ARPU на `ax2 = ax1.twinx()` линией. Обязательно свяжите цвета: `ax1.tick_params(axis="y", labelcolor="C0")`, `ax2.tick_params(axis="y", labelcolor="C1")`. Настройте `FuncFormatter` для ARPU (`f"{v:,.0f} ₽"`) и посещений (`f"{v:.0f}"`). Соберите общую легенду через `get_legend_handles_labels()`. Ожидаемый вывод: посещения и ARPU на одной оси с разными шкалами; цвета осей совпадают с цветом данных. В комментариях объясните: зачем `twinx` (несопоставимые масштабы — без него ARPU «схлопнется» в плоскую линию на фоне посещений); почему **обязательно** связывать цвет данных и осей; как `get_legend_handles_labels()` собирает элементы с обеих осей и почему `ax1.legend()` покажет только линии `ax1`; когда `twiny` (вторая ось X) уместнее — например, две шкалы по времени (дни и недели); как `ax2.spines["right"].set_color("C1")` красит правую рамку в цвет данных; что произойдёт без `set_ylim` на каждой оси — Matplotlib сам подберёт диапазон, но иногда начнёт с нуля и «сплющит» ARPU (3000 и 4000 ₽ будут неразличимы).

**Задача 22 (box plot из read_csv).** Фитнес-клуб сравнивает стоимость персональной тренировки по тренерам (Иванов, Петров, Смирнова, Сидоров). Постройте box plot двумя способами: через `df.boxplot(column="price_rub", by="trainer", grid=False, patch_artist=True)` (обёртка Pandas) и вручную через `groups = [g["price_rub"].values for _, g in df.groupby("trainer")]; ax.boxplot(groups, tick_labels=names, patch_artist=True, medianprops=dict(color="red", linewidth=2))`. Ожидаемый вывод: одинаковые графики двумя разными способами. В комментариях объясните: как `df.boxplot(by=...)` связан с Matplotlib (это просто обёртка, которая вызывает `ax.boxplot` под капотом); зачем `grid=False`; что такое `notch=True` (вырез для доверительного интервала медианы — если «талии» двух ящиков не пересекаются, медианы различаются статистически); почему `patch_artist=True` нужен для закраски ящиков и без него `facecolor` игнорируется; почему в новых версиях Matplotlib (3.9+) параметр `labels` устарел в пользу `tick_labels`; что произойдёт при малой выборке (5–10 значений) — box plot станет малоинформативным, лучше `strip plot` (`ax.scatter` с jitter).

---



## 🟣 Уровень 3. Мастер (задачи 23–32)

**Задача 23 (FigureBuilder — ООП-обёртка для сетки).** Фитнес-клуб готовит дашборд из 4 графиков (динамика посещаемости, топ тренеров, распределение длительности, выручка по филиалам). При построении код обрастает повторяющимися `axes[i, j].plot(...)` и `axes[i, j].set_title(...)`. Реализуйте `class FigureBuilder` с `__init__(self, nrows=1, ncols=1, **kw)`, методами `plot(i, j, x, y, **kw)`, `title(i, j, text, **kw)`, `grid(**kw)` и `savefig(path, **kw)`, возвращающими `self` для цепочек. Используйте `np.atleast_2d(self.axes)`, чтобы индексация `[i, j]` работала и для одномерных сеток. Создайте `FigureBuilder(2, 2, figsize=(12, 8), layout="constrained").plot(0, 0, ...).title(0, 0, "Посещаемость").grid(alpha=0.3).savefig("dashboard.png", dpi=300)`. Ожидаемый вывод: сетка 2×2 с 4 графиками. В комментариях объясните: зачем fluent API (читаемость цепочек — код похож на английский); что произойдёт при `subplots(1, 3)` без `atleast_2d` (одномерный массив — индексация `[0, 1]` выбросит `IndexError`); как расширить класс методами `set_xlabel`, `legend`, `axhline`, чтобы сохранить принцип единственной ответственности; когда такой класс — овер-инжиниринг; как связать `FigureBuilder` с `PlotConfig` из задачи 2; почему `return self` в каждом методе — ключевой паттерн (позволяет цепочки любой длины).


In [ ]:
# --- генерируем данные для дашборда ---
np.random.seed(123)
dates = pd.date_range("2024-01-01", periods=90, freq="D")
visits = 150 + np.cumsum(np.random.randint(-5, 6, size=90))
trainers = ["Иванов", "Смирнова", "Кузнецов", "Попова", "Соколов"]
trainer_counts = np.array([180, 150, 130, 110, 90])
durations = np.concatenate([
    np.random.normal(50, 8, 200),
    np.random.normal(70, 10, 150),
    np.random.normal(90, 7, 80),
])
branches = ["Центральный", "Северный", "Южный", "Западный"]
revenue = np.array([420_000, 380_000, 290_000, 210_000])

# --- FigureBuilder ---
class FigureBuilder:
    def __init__(self, nrows=1, ncols=1, **kw):
        self.fig, self.axes = plt.subplots(nrows, ncols, **kw)
        self.axes = np.atleast_2d(self.axes)

    def plot(self, i, j, x, y, **kw):
        self.axes[i, j].plot(x, y, **kw)
        return self

    def bar(self, i, j, x, y, **kw):
        self.axes[i, j].bar(x, y, **kw)
        return self

    def hist(self, i, j, x, **kw):
        self.axes[i, j].hist(x, **kw)
        return self

    def title(self, i, j, text, **kw):
        self.axes[i, j].set_title(text, **kw)
        return self

    def xlabel(self, i, j, text, **kw):
        self.axes[i, j].set_xlabel(text, **kw)
        return self

    def ylabel(self, i, j, text, **kw):
        self.axes[i, j].set_ylabel(text, **kw)
        return self

    def legend(self, i, j, **kw):
        self.axes[i, j].legend(**kw)
        return self

    def axhline(self, i, j, y, **kw):
        self.axes[i, j].axhline(y, **kw)
        return self

    def grid(self, **kw):
        for ax in self.axes.flat:
            ax.grid(**kw)
        return self

    def savefig(self, path, **kw):
        self.fig.savefig(path, **kw)
        return self

(
    FigureBuilder(2, 2, figsize=(12, 8), layout="constrained")
    .plot(0, 0, dates, visits, color="steelblue")
    .title(0, 0, "Посещаемость")
    .xlabel(0, 0, "Дата")
    .ylabel(0, 0, "Визитов")
    .bar(0, 1, trainers, trainer_counts, color="seagreen")
    .title(0, 1, "Топ тренеров")
    .hist(1, 0, durations, bins=20, color="orange", edgecolor="black")
    .title(1, 0, "Длительность тренировок")
    .xlabel(1, 0, "Минуты")
    .bar(1, 1, branches, revenue, color="crimson")
    .title(1, 1, "Выручка по филиалам")
    .ylabel(1, 1, "руб.")
    .grid(alpha=0.3)
    .savefig("dashboard.png", dpi=300, bbox_inches="tight")
)

# Комментарии:
# Fluent API — методы возвращают self, и вызовы выстраиваются в цепочку,
# читаемую "как английский": построить → озаглавить → сетка → сохранить.
# Это снижает шум (нет повторяющихся ax = axes[i, j]).
# Без np.atleast_2d для subplots(1, 3) axes — одномерный массив;
# индексация axes[0, 1] выбросит IndexError (ndarray 1-D ждёт один
# индекс). atleast_2d превращает в форму (1, 3) — [0, j] работает.
# Расширение класса: set_xlabel/set_ylabel/legend/axhline — каждый
# метод делает ОДНУ вещь для ОДНОЙ оси. Принцип единственной
# ответственности сохраняется, цепочка растёт органично.
# Овер-инжиниринг: для 1–2 графиков и одноразового скрипта класс
# избыточен — plt.subplots + несколько вызовов быстрее. FigureBuilder
# оправдан при повторяемых дашбордах и большом числе панелей.
# Связь с PlotConfig: можно принять PlotConfig в __init__ и передать
# figsize/dpi/facecolor/layout в plt.subplots — конфиг остаётся
# единственным источником стиля, builder — единственным "сборщиком".
# return self в каждом методе — ключевой паттерн fluent API: цепочка
# может быть любой длины и прерывается вызовом savefig/show.

plt.show()
print()


**Задача 24 (GridSpec и subplot_mosaic).** Фитнес-клуб строит дашборд с большим графиком динамики посещаемости сверху и двумя меньшими снизу (топ тренеров и выручка по филиалам). Реализуйте двумя способами: через `gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], width_ratios=[3, 1])` с `ax1 = fig.add_subplot(gs[0, :])`, `ax2 = fig.add_subplot(gs[1, 0])`, `ax3 = fig.add_subplot(gs[1, 1])`; и через `fig, axes = plt.subplot_mosaic("AAA;BC", figsize=(12, 8), layout="constrained")` с обращением `axes["A"]`, `axes["B"]`, `axes["C"]`. Ожидаемый вывод: верхний график во всю ширину в 2× выше нижних. В комментариях объясните: как `gs[0, :]` означает «вся первая строка» (срез по NumPy), а `gs[1, 0]` — конкретная ячейка; зачем `height_ratios=[2, 1]` и `width_ratios=[3, 1]`; почему `subplot_mosaic` читается лучше для сложных сеток («AAA;BC» — почти как ASCII-арт); как `GridSpecFromSubplotSpec` создаёт вложенные сетки; что произойдёт при пересечении областей (например, `gs[0, :2]` и `gs[0, 1:]` — наложение, Matplotlib выдаст предупреждение); как сохранить `layout="constrained"` с `GridSpec` (передать в `figure(layout="constrained")`).

In [ ]:
# --- данные для дашборда ---
np.random.seed(124)
dates = pd.date_range("2024-01-01", periods=90, freq="D")
visits = 150 + np.cumsum(np.random.randint(-5, 6, size=90))
trainers = ["Иванов", "Смирнова", "Кузнецов", "Попова", "Соколов"]
trainer_counts = np.array([180, 150, 130, 110, 90])
branches = ["Центральный", "Северный", "Южный", "Западный"]
revenue = np.array([420_000, 380_000, 290_000, 210_000])

# --- способ 1: GridSpec ---
fig = plt.figure(figsize=(12, 8), layout="constrained")
gs = fig.add_gridspec(2, 2, height_ratios=[2, 1], width_ratios=[3, 1])

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])

ax1.plot(dates, visits, color="steelblue")
ax1.set_title("Динамика посещаемости (gs[0, :])")
ax1.grid(alpha=0.3)

ax2.barh(trainers[::-1], trainer_counts[::-1], color="seagreen")
ax2.set_title("Топ тренеров")

ax3.bar(branches, revenue, color="crimson")
ax3.set_title("Выручка")
ax3.tick_params(axis="x", rotation=30)

fig.savefig("dashboard_gridspec.png", dpi=150, bbox_inches="tight")

# --- способ 2: subplot_mosaic (все строки одинаковой длины) ---
fig2, axes = plt.subplot_mosaic(
    "AAA;BC.", figsize=(12, 8), layout="constrained"
)

axes["A"].plot(dates, visits, color="steelblue")
axes["A"].set_title("Динамика посещаемости (мозаика A)")
axes["A"].grid(alpha=0.3)

axes["B"].barh(trainers[::-1], trainer_counts[::-1], color="seagreen")
axes["B"].set_title("Топ тренеров")

axes["C"].bar(branches, revenue, color="crimson")
axes["C"].set_title("Выручка")
axes["C"].tick_params(axis="x", rotation=30)

fig2.savefig("dashboard_mosaic.png", dpi=150, bbox_inches="tight")

# --- вложенная сетка GridSpecFromSubplotSpec ---
fig3 = plt.figure(figsize=(12, 8), layout="constrained")
outer = fig3.add_gridspec(2, 2, height_ratios=[2, 1])
ax_top = fig3.add_subplot(outer[0, :])
ax_top.plot(dates, visits)
ax_top.set_title("Верх: общая динамика")

inner = outer[1, 0].subgridspec(1, 2, width_ratios=[2, 1])
ax_in1 = fig3.add_subplot(inner[0, 0])
ax_in2 = fig3.add_subplot(inner[0, 1])
ax_in1.bar(trainers, trainer_counts)
ax_in1.set_title("Вложенная: тренеры")
ax_in2.bar(branches, revenue)
ax_in2.set_title("Вложенная: выручка")

ax_side = fig3.add_subplot(outer[1, 1])
ax_side.pie(revenue, labels=branches, autopct="%1.0f%%")
ax_side.set_title("Доли филиалов")

# Комментарии:
# gs[0, :] — NumPy-срез: вся первая строка (все столбцы). gs[1, 0] —
# конкретная ячейка (вторая строка, первый столбец). GridSpec
# возвращает "слоты", которые add_subplot превращает в Axes.
# height_ratios=[2, 1] — верхняя строка в 2 раза выше нижней
# (большой график динамики + два маленьких). width_ratios=[3, 1] —
# в нижней строке левый график втрое шире правого.
# subplot_mosaic("AAA;BC.") — ASCII-арт: строка "AAA" — верх на всю
# ширину, "BC." — B и C в нижних двух ячейках, "." — пустая
# (C не растягивается до края). Строки ОБЯЗАНЫ быть одинаковой длины,
# иначе ValueError: All of the rows must be the same length.
# .subgridspec(...) (то же, что GridSpecFromSubplotSpec) строит
# вложенные сетки внутри ячейки — не переделывая внешнюю сетку.
# Пересечение областей (gs[0, :2] и gs[0, 1:]) — Matplotlib выдаст
# UserWarning об overlapping axes; результат неопределён — избегать.
# layout="constrained" работает и с GridSpec, и с subplot_mosaic —
# передаём в figure(...)/subplot_mosaic(...), он сам разводит
# подписи, титулы, легенды.

plt.show()
print()



**Задача 25 (sharex/sharey и синхронизация зума).** Фитнес-клуб строит два подграфика: дневная посещаемость и её первая разность (прирост день ко дню). Обе линии относятся к одним и тем же датам, и хочется синхронизировать зум. Создайте `fig, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 8), layout="constrained")`, постройте `axes[0].plot(df.index, df["visits"])` и `axes[1].plot(df.index, df["visits"].diff())`, уберите подпись X у верхнего (`axes[0].set_xlabel("")`). Дополнительно создайте вторую фигуру **без** `layout` и примените `fig.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.05)` — почти вплотную. Ожидаемый вывод: две синхронизированные оси X; без дублирования подписи. В комментариях объясните: как `sharex` синхронизирует зум (перетаскивание одной оси двигает обе); зачем убирать подпись X у верхнего подграфика (визуальный шум — даты повторяются); почему `subplots_adjust` **не работает** с `layout="constrained"`; что делает `hspace=0.05`; как `ax.get_xlim()` получить текущие границы для сохранения в отчёт; что произойдёт при `sharex="col"` в сетке 2×2.


In [ ]:
# --- данные ---
np.random.seed(125)
dates = pd.date_range("2024-01-01", periods=90, freq="D")
visits = 150 + np.cumsum(np.random.randint(-5, 6, size=90))
dfsh = pd.DataFrame({"visits": visits}, index=dates)

# --- вариант 1: sharex + constrained ---
fig, axes = plt.subplots(
    2, 1, sharex=True, figsize=(12, 8), layout="constrained"
)
axes[0].plot(dfsh.index, dfsh["visits"], color="steelblue")
axes[0].set_ylabel("Посещаемость")
axes[0].set_xlabel("")  # убираем дублирование подписи X

axes[1].plot(dfsh.index, dfsh["visits"].diff(),
             color="crimson", marker="o", markersize=3)
axes[1].set_ylabel("Прирост день ко дню")
axes[1].set_xlabel("Дата")

fig.savefig("visits_sharex.png", dpi=150, bbox_inches="tight")

# --- вариант 2: без constrained + subplots_adjust ---
fig2, axes2 = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
axes2[0].plot(dfsh.index, dfsh["visits"], color="steelblue")
axes2[0].set_xlabel("")
axes2[1].plot(dfsh.index, dfsh["visits"].diff(), color="crimson")

fig2.subplots_adjust(
    left=0.1, right=0.9, top=0.9, bottom=0.1, hspace=0.05
)
fig2.savefig("visits_sharex_adjust.png", dpi=150, bbox_inches="tight")

# --- пример: снять текущие границы для отчёта ---
xlim = axes[1].get_xlim()
print("Текущий xlim (числовой):", xlim)

# Комментарии:
# sharex=True связывает оси X: зум/панорамирование на одной оси
# автоматически применяется к другой (общий ViewLim/Transform).
# Убираем подпись X у верхнего графика: даты одинаковые, дублирование
# подписи — визуальный шум; нижний график несёт подпись как "основную".
# subplots_adjust не работает с layout="constrained": constrained
# layout сам управляет отступами (через layout engine), значения
# left/right/top/bottom/hspace ИГНОРИРУЮТСЯ. Нужно либо отключить
# layout (subplots без layout), либо использовать constrained.
# hspace=0.05 — почти вплотную между подграфиками: полезно для
# "сшитых" графиков, где важно визуально сопоставить линии.
# ax.get_xlim() возвращает текущие границы в числовом формате
# (дни с эпохи для дат — числа с плавающей точкой). Для отчёта можно
# перевести через mdates.num2date(...) или просто зафиксировать.
# sharex="col" в сетке 2×2 синхронизирует ось X ПО СТОЛБЦАМ:
# верхняя и нижняя оси каждого столбца делят X, но разные столбцы —
# независимы. Удобно, когда в столбце — временной ряд и его
# производная, а столбцы — разные метрики.

plt.show()
print()


**Задача 26 (bar с yerr из agg(["mean", "std"])).** Фитнес-клуб сравнивает среднюю стоимость тренировки по типам занятий с учётом разброса между клиентами. Получите `stats = results.groupby("class_type")["price_rub"].agg(["mean", "std"])`. Постройте `bars = ax.bar(stats.index, stats["mean"], yerr=stats["std"], capsize=8, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(stats))), edgecolor="black")`. Установите `ax.set_ylim(0, stats["mean"].max() * 1.3)`, подпишите значения через `for bar, (m, s) in zip(bars, stats.itertuples(index=False)): ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + s + 50, f"{m:,.0f}", ha="center")`. Ожидаемый вывод: столбцы с планками ошибок и подписями. В комментариях объясните: как `agg(["mean", "std"])` готовит данные в одну строку (что было бы без `agg` — пришлось бы дважды вызывать `groupby`); почему `yerr=std` корректен (это стандартное отклонение, а не SEM; для SEM поделите на `sqrt(n)`); почему `set_ylim` с запасом `* 1.3` (чтобы планки и подписи не обрезались); что произойдёт без `capsize`; как `zip(bars, stats.itertuples())` обходит две коллекции одновременно; почему для метрик-долей `set_ylim(0, 1)` обязателен.

In [ ]:
# --- генерируем results.csv ---
np.random.seed(126)
classes = ["йога", "пилатес", "кроссфит", "бассейн", "бокс"]
base_price = {"йога": 800, "пилатес": 900, "кроссфит": 1100,
              "бассейн": 700, "бокс": 1000}
rows = []
for ct in classes:
    for _ in range(60):
        rows.append({
            "class_type": ct,
            "price_rub": max(int(base_price[ct] + np.random.normal(0, 120)), 0),
        })
pd.DataFrame(rows).to_csv("results.csv", index=False)

# --- загрузка ---
results = pd.read_csv("results.csv")

stats = results.groupby("class_type")["price_rub"].agg(["mean", "std"])
n = results.groupby("class_type")["price_rub"].count()
sem = stats["std"] / np.sqrt(n)

fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
bars = ax.bar(
    stats.index, stats["mean"],
    yerr=stats["std"], capsize=8,
    color=plt.cm.viridis(np.linspace(0.2, 0.8, len(stats))),
    edgecolor="black",
)

ax.set_ylim(0, stats["mean"].max() * 1.3)
for bar, (m, s) in zip(bars, stats.itertuples(index=False)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + s + 50,
        f"{m:,.0f}", ha="center",
    )

ax.set_ylabel("Средняя цена, руб.")
ax.set_title("Средняя стоимость тренировки по типам (± std)")

# --- вариант с SEM ---
fig2, ax2 = plt.subplots(figsize=(10, 6), layout="constrained")
bars2 = ax2.bar(
    stats.index, stats["mean"], yerr=sem, capsize=8,
    color="steelblue", edgecolor="black",
)
ax2.set_ylim(0, stats["mean"].max() * 1.3)
for bar, (m, s) in zip(bars2, stats.itertuples(index=False)):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + s + 50,
        f"{m:,.0f}", ha="center",
    )
ax2.set_title("Средняя цена (± SEM)")

# Комментарии:
# agg(["mean", "std"]) в один вызов даёт DataFrame со столбцами mean
# и std — не надо дважды вызывать groupby(...)["price"].mean() и
# .std(). Один проход, согласованный порядок индексов.
# yerr=std корректен, если нужен РАЗБРОС КЛИЕНТОВ (стандартное
# отклонение цен). Если нужна точность средней — используйте SEM =
# std / sqrt(n); она всегда меньше std и показывает "неопределённость
# среднего", а не разброс.
# set_ylim(..., mean.max()*1.3) — запас под планки ошибок и подписи
# значений. Без запаса planks/тексты срежутся на верхней границе.
# Без capsize "шапочки" на концах усов будут отсутствовать, а сам
# масштаб (длина) — тот же; capsize лишь визуально фиксирует ширину
# горизонтальных перекладин.
# zip(bars, stats.itertuples(index=False)) идёт по двум коллекциям
# параллельно: bar — BarContainer-элемент, кортеж (mean, std) —
# строка stats без индекса. Удобно без ручных индексов.
# Для метрик-долей (конверсия, доля клиентов) set_ylim(0, 1)
# обязателен: иначе ось может "обрезать" 1.0 или начинаться не с 0 —
# столбцы визуально искажаются.

plt.show()
print()



**Задача 27 (scatter + RANSAC-тренд).** Фитнес-клуб анализирует связь числа тренеров в смене и времени ожидания клиента, но в данных есть несколько дней с аномальными результатами (сбой системы записи, наплыв новичков в январе) — они искажают картину. Обычная линейная регрессия через `np.polyfit` сдвигается этими выбросами. Реализуйте два тренда: OLS через `m, b = np.polyfit(x, y, 1)` и RANSAC через `RANSACRegressor(random_state=42).fit(X, y)`. Наложите оба: OLS — `"r--"`, RANSAC — `"g-"`, подпишите в легенде. Вычислите `r_pearson = np.corrcoef(x, y)[0, 1]` и `r_spearman = scipy.stats.spearmanr(x, y).correlation`. В комментариях объясните: чем OLS отличается от RANSAC; как интерпретировать разницу между `r_pearson` и `r_spearman` (нелинейность, выбросы — если Пирсон «портится», а Спирмен стабилен, значит, выбросы); почему `polyfit(deg=2)` для нелинейных зависимостей (например, эффект «насыщения» — после N тренеров ожидание падает нелинейно); как `zorder` держит RANSAC-линию поверх точек; что произойдёт при полном отсутствии выбросов (RANSAC ≈ OLS); как сохранить `ransac.estimator_` — коэффициенты найденной модели для отчёта.

**Задача 28 (contour функции потерь).** Фитнес-клуб использует ML для прогнозирования оттока клиентов. Чтобы понять, насколько сложна оптимизация модели, визуализируйте функцию потерь от двух параметров (например, `w1` = порог скидки, `w2` = коэффициент лояльности). Используйте функцию Розенброка `loss(w1, w2) = (w1 - 1)**2 + 10 * (w2 - w1**2)**2`. Создайте `meshgrid` 200×200, постройте `cf = ax.contourf(W1, W2, np.log1p(Z), levels=20, cmap="viridis")` и поверх `ax.contour(W1, W2, np.log1p(Z), levels=15, colors="white", linewidths=0.5)`. Отметьте минимум `ax.plot(1, 1, "r*", markersize=20)`. Ожидаемый вывод: «долина Розенброка» с изолиниями и минимумом. В комментариях объясните: зачем `np.log1p(Z)` (сжать динамический диапазон — иначе вся «долина» будет одного цвета); почему для несимметричных данных выбирают sequential-палитру, а для симметричных относительно нуля — diverging (`RdBu`); что делает `clabel(contour, inline=True)`; как `levels=np.logspace(-1, 3, 20)` помогает для данных, меняющихся в тысячи раз; что произойдёт при `levels=3` (теряется форма); как `norm="log"` в `contourf` автоматически применяет логарифмическую шкалу.

**Задача 29 (heatmap корреляций).** Фитнес-клуб изучает, какие признаки сильнее всего коррелируют с оттоком клиентов (цена, число тренировок, рейтинг тренера, длительность абонемента, возраст, посещаемость). Постройте `corr = df.select_dtypes("number").corr(method="spearman")` (ранговая корреляция устойчива к выбросам), затем `im = ax.imshow(corr, cmap="RdBu", vmin=-1, vmax=1)` с `fig.colorbar(im, label="Spearman ρ")`. Скройте верхнюю половину через маску `mask = np.triu(np.ones_like(corr, dtype=bool), k=1); corr_masked = corr.mask(mask)`. Аннотируйте значения через `for i, j in zip(*np.where(~mask)): ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")`. Ожидаемый вывод: матрица 6×6 с диагональю 1.0 и цветовой шкалой. В комментариях объясните: почему `vmin=-1, vmax=1` фиксирует диапазон (нуль всегда белый, иначе цвет «плавает» в зависимости от данных); зачем **diverging**-палитра `RdBu` для корреляций; как `select_dtypes("number")` защищает от ошибок (строковые столбцы типа «тип абонемента» не коррелируют); почему `method="spearman"` предпочтительнее `pearson` для реальных операционных данных; как `np.where(~mask)` даёт координаты только нижнего треугольника; как отсортировать признаки по корреляции с целевой (`corr.abs().sort_values(by="churn", ascending=False)`) для приоритизации.

**Задача 30 (box plot с notch и сравнение с violinplot).** Фитнес-клуб сравнивает распределение длительности тренировок по типам занятий. Box plot компактен, но **скрывает** мультимодальность — а у кроссфита длительность может быть бимодальной (короткие WOD vs длинные комплексы). Постройте `bp = ax.boxplot(groups, tick_labels=names, patch_artist=True, notch=True, showmeans=True, medianprops=dict(color="red", linewidth=2), meanprops=dict(marker="D", markerfacecolor="yellow", markeredgecolor="black", markersize=10))`. Рядом постройте `ax.violinplot(groups, showmedians=True, showextrema=True)`. Ожидаемый вывод: box plot с «талией» и жёлтыми ромбами средних рядом с violin plot, где видна бимодальность у кроссфита. В комментариях объясните: что показывает `notch` (доверительный интервал медианы — если «талии» двух ящиков не пересекаются, медианы статистически различаются); зачем `showmeans`, если уже есть медиана; почему `violinplot` информативнее box plot (видна плотность — два «горба» у кроссфита сразу заметны); когда box plot лучше (мало данных, много групп — компактность); как наложить точки через `ax.scatter` (strip plot) поверх box plot; что произойдёт при `notch=True` на малой выборке (Matplotlib выведет предупреждение о ненадёжности).

**Задача 31 (errorbar + fill_between для CI + resample).** Фитнес-клуб мониторит недельную посещаемость с доверительным интервалом. Сгруппируйте по неделям: `weekly = measurements.resample("W", on="date")["visits"].agg(["mean", "std", "count"])`. Вычислите `sem = weekly["std"] / np.sqrt(weekly["count"])`, `ci = 1.96 * sem` (95% CI для нормального распределения). Постройте `ax.errorbar(weekly.index, weekly["mean"], yerr=ci, fmt="o-", capsize=6, color="steelblue")` и `ax.fill_between(weekly.index, weekly["mean"] - ci, weekly["mean"] + ci, alpha=0.15, color="steelblue")`. Ожидаемый вывод: среднее с планками ошибок и той же полосой CI. В комментариях объясните: чем SD отличается от SEM (первое — разброс наблюдений, второе — точность оценки среднего; SEM всегда меньше SD при `n > 1`); зачем `1.96 * sem` (95% CI для нормального распределения); когда `fill_between` читается лучше `errorbar` (плотные ряды — планки сливаются в «частокол»); как `resample("W")` группирует по неделям и почему нужен `DatetimeIndex`; что произойдёт при `count = 1` в неделе (`sem = 0/1 = 0`, CI сжимается до точки); как использовать `t`-распределение для малых выборок (`scipy.stats.t.ppf(0.975, df=count-1)` вместо 1.96).

**Задача 32 (форматирование чисел: собственные Formatter-классы).** Фитнес-клуб готовит финансовый отчёт: выручка доходит до 1 млрд руб., конверсия измеряется в процентах, а число клиентов — в тысячах. Стандартные форматтеры выглядят «сыро». Реализуйте три класса: `class CompactFormatter(mticker.Formatter)` (`1500000 → "1.5M"`, `250000 → "250K"`), `class RubleFormatter` (`1500000 → "1 500 000 ₽"` через `f"{x:,.0f} ₽".replace(",", " ")`), `class PercentFormatterCustom` (`0.85 → "85.0%"`). Сравните с готовыми `EngFormatter(unit="₽")` и `PercentFormatter(1.0)`. Примените к трём подграфикам. Ожидаемый вывод: разные форматы на одной и той же оси Y. В комментариях объясните: зачем ООП-класс вместо `FuncFormatter`; как `Formatter` встраивается в `Locator` и почему это разделение ответственности; почему `PercentFormatter(1.0)` ожидает доли (0.25 → 25%), а не проценты (что было бы при `PercentFormatter(100)`); когда `StrMethodFormatter("{x:.2f}")` проще класса; как `EngFormatter` использует инженерные приставки (k, M, G) и когда это удобнее «M/K»; как сохранить единый формат в отчёте через `plt.rcParams` (нет стандартного параметра — нужна своя функция-обёртка).



---

## 🟠 Уровень 4. Архитектор (задачи 33–42)

**Задача 33 (форматирование дат через mdates).** Фитнес-клуб готовит график сезонности посещаемости за 3 года. «Сырые» даты выглядят как числа дней с 1970 года — нечитаемо. Освойте три стратегии: (а) `ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))` + `DateFormatter("%b %Y")` (раз в квартал); (б) `ax.xaxis.set_major_locator(mdates.AutoDateLocator())` + `AutoDateFormatter(...)` (автоматический подбор); (в) `ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))` + `DateFormatter("%d.%m (%a)")` (только понедельники). Для каждого поверните метки через `plt.setp(ax.get_xticklabels(), rotation=45, ha="right")`. Ожидаемый вывод: три подграфика с разными форматами дат. В комментариях объясните: зачем **два** инструмента (Locator решает «где», Formatter «как» — без Locator форматер поставит деления слишком часто); почему `parse_dates` при `read_csv` критичен; чем `%b` отличается от `%B` (`Jan` vs `January`); почему `AutoDateLocator` — хороший дефолт, если не хочется думать о деталях; что произойдёт при `MonthLocator(interval=12)` на 3-летнем ряду (всего 3 деления — неинформативно); как `mdates.YearLocator(base=5)` для долгосрочных данных.

In [ ]:
# --- данные: 3 года ежедневной посещаемости с сезонностью ---
np.random.seed(133)
dates3 = pd.date_range("2022-01-01", "2024-12-31", freq="D")
t = np.arange(len(dates3))
season = 40 * np.sin(2 * np.pi * t / 365.25 - np.pi / 2)  # пик летом
trend = 0.02 * t
noise = np.random.normal(0, 10, size=len(dates3))
visits3 = np.clip(180 + season + trend + noise, 30, None)
df3 = pd.DataFrame({"visits": visits3}, index=dates3)

fig, axes = plt.subplots(3, 1, figsize=(12, 10), layout="constrained")
strategies = [
    ("(а) MonthLocator(interval=3) + %b %Y", "quarter"),
    ("(б) AutoDateLocator + AutoDateFormatter", "auto"),
    ("(в) WeekdayLocator(MO) + %d.%m (%a)", "weekday"),
]

for ax, (title, kind) in zip(axes, strategies):
    ax.plot(df3.index, df3["visits"], color="steelblue", linewidth=1)
    if kind == "quarter":
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    elif kind == "auto":
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        ax.xaxis.set_major_formatter(mdates.AutoDateFormatter(
            mdates.AutoDateLocator()))
    else:
        # для наглядности — один месяц, чтобы понедельники не слипались
        ax.set_xlim(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31"))
        ax.xaxis.set_major_locator(
            mdates.WeekdayLocator(byweekday=mdates.MO))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m (%a)"))
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

fig.savefig("seasonality_dates.png", dpi=150, bbox_inches="tight")

# --- YearLocator(base=5) для долгосрочных данных ---
fig2, ax2 = plt.subplots(figsize=(12, 4), layout="constrained")
ax2.plot(df3.index, df3["visits"])
ax2.xaxis.set_major_locator(mdates.YearLocator(base=1))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.set_title("YearLocator(base=1): по годам")
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

# Комментарии:
# Два инструмента: Locator решает ГДЕ ставить тики (позиции),
# Formatter — КАК их подписать. Без Locator DateFormatter расставит
# подписи на дефолтных позициях (AutoDateLocator), которые могут
# быть слишком частыми/редкими.
# parse_dates при read_csv критичен: без него колонка дат — строки
# (или int), Matplotlib не сможет применить mdates-локаторы, а ось X
# будет числами. С parse_dates + index_col даты — DatetimeIndex.
# %b — сокращённое название месяца (Jan, Feb), %B — полное
# (January, February). Для компактных меток лучше %b.
# AutoDateLocator — хороший дефолт: сам выбирает шаг (год/месяц/
# день) в зависимости от масштаба и зума; AutoDateFormatter
# подбирает формат под этот шаг. Идеально, когда нет требований
# к конкретному формату.
# MonthLocator(interval=12) на 3-летнем ряду даст всего 3 деления —
# по одному в год, слишком редко для сезонности; для 3 лет оптимально
# interval=3 (квартал).
# YearLocator(base=5) — раз в 5 лет: для 20-летних трендов, чтобы
# не засорять ось сотнями меток. base=1 — каждый год.

plt.show()
print()



**Задача 34 (операционный дашборд фитнес-клуба с twinx + mdates + Formatter).** Фитнес-клуб мониторит число посещений и средний чек для управляющей компании. Постройте профессиональный график: (а) число посещений на `ax1` линией; (б) средний чек на `ax2 = ax1.twinx()`; (в) скользящее среднее посещений `df["visits"].rolling(7).mean()` на `ax1`. Настройте `ax1.xaxis.set_major_locator(mdates.MonthLocator())`, `DateFormatter("%b %Y")`, `ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:.0f}"))`, `ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, p: f"{v:,.0f} ₽"))`. Соберите общую легенду. Ожидаемый вывод: посещения + SMA(7) на главной оси, средний чек на правой, даты по месяцам. В комментариях объясните: зачем `rolling(7)` (сглаживает дневной шум — недельный тренд); как `line` на `twinx` не мешает основной линии (визуальное разделение через цвет); почему цвета (синий/оранжевый) должны быть различимы даже для дальтоников (используйте `#1f77b4`/`#ff7f0e` — они хорошо видны); как `width=0.8` в `bar` оставляет зазоры между днями (или `width=1` для плотной заливки); как сохранить для веба (`dpi=150`) и для печати (`dpi=300`) — разные требования к размеру; что произойдёт без `twinx` (средний чек «схлопнется» на фоне посещений — шкала 3000 ₽ vs 500).


In [ ]:
# --- данные: 180 дней, посещения и средний чек ---
np.random.seed(134)
dates6 = pd.date_range("2024-01-01", periods=180, freq="D")
t = np.arange(180)
visits = 180 + 25 * np.sin(2 * np.pi * t / 30) + np.random.normal(0, 15, 180)
check = 1200 + 150 * np.sin(2 * np.pi * t / 45 + 0.5) + np.random.normal(0, 60, 180)
dfo = pd.DataFrame({"visits": np.clip(visits, 30, None),
                    "avg_check": np.clip(check, 200, None)}, index=dates6)

fig, ax1 = plt.subplots(figsize=(13, 6), layout="constrained")

# главная ось: посещения + SMA(7)
c_vis = "#1f77b4"
c_chk = "#ff7f0e"
ax1.plot(dfo.index, dfo["visits"], color=c_vis, alpha=0.4,
         linewidth=1, label="Посещения (день)")
ax1.plot(dfo.index, dfo["visits"].rolling(7).mean(),
         color=c_vis, linewidth=2.5, label="SMA(7) посещений")
ax1.set_ylabel("Посещения", color=c_vis)
ax1.tick_params(axis="y", labelcolor=c_vis)

# правая ось: средний чек
ax2 = ax1.twinx()
ax2.plot(dfo.index, dfo["avg_check"], color=c_chk,
         linewidth=1.5, label="Средний чек")
ax2.set_ylabel("Средний чек, ₽", color=c_chk)
ax2.tick_params(axis="y", labelcolor=c_chk)

# форматирование дат
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(ax1.get_xticklabels(), rotation=45, ha="right")

# форматирование значений
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, p: f"{v:.0f}"))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, p: f"{v:,.0f} ₽"))

# объединённая легенда
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="upper left", framealpha=0.9)

ax1.set_title("Операционный дашборд: посещения и средний чек")

# два сохранения — веб и печать
fig.savefig("dashboard_web.png", dpi=150, bbox_inches="tight")
fig.savefig("dashboard_print.png", dpi=300, bbox_inches="tight")

# Комментарии:
# rolling(7) — скользящее среднее за 7 дней: сглаживает дневной шум
# (выходные/будни, случайные всплески), оставляя недельный тренд.
# ax1.twinx() создаёт вторую ось Y, делящую ось X: линии НЕ мешают
# друг другу (у каждой своя шкала Y). Если бы обе метрики были на
# одной оси, средний чек (~1500) визуально "схлопнулся" бы под
# посещениями (~180) — шкалы несопоставимы.
# Цвета #1f77b4 / #ff7f0e — пара из палитры Matplotlib, различимая
# при протанопии (красно-зелёной слепоте); избегайте красный/зелёный
# для главных осей.
# width в bar (если бы использовали bar): width=0.8 оставляет зазор
# между столбцами (дни чётко разделены), width=1 — плотная заливка
# (непрерывный ряд, но столбцы сливаются).
# dpi=150 для веба (быстрая загрузка, экраны), dpi=300 для печати
# (полиграфия); сохраняем две версии.
# Без twinx средний чек и посещения делят ось — масштаб 1500 vs 180
# делает одну из линий почти плоской у нижнего края.

plt.show()
print()


**Задача 35 (Figure3D — подкласс-обёртка для 3D-сцен).** Фитнес-клуб визуализирует 3D-связь возраста, веса и числа тренировок в неделю (облако точек), а также спираль тренда (если данные позволяют). 3D в Matplotlib создаётся через `plt.subplots(subplot_kw={"projection": "3d"})`, возвращающий `Axes3D`. Реализуйте `class Figure3D` с методами `helix(n=200)` (спираль через `ax.plot(np.cos(t), np.sin(t), t)`), `cloud(n=500)` (облако через `ax.scatter(x, y, z, c=z, cmap="viridis", s=20)`), `view(elev=30, azim=-60)` (угол обзора). Постройте спираль и облако, вызовите `view(20, -45)`. Ожидаемый вывод: 3D-сцена с возможностью вращения мышью. В комментариях объясните: зачем класс-обёртка (переиспользование, fluent API); как `projection="3d"` создаёт `Axes3D` — подкласс обычного `Axes`; что делает `view_init(elev, azim)` (углы обзора: `elev` — подъём 0–90°, `azim` — поворот 0–360°); как `depthshade=True` (по умолчанию) затемняет дальние точки, создавая иллюзию глубины; почему 3D сложнее читать, чем 2D (перспективные искажения, перекрытия — лучше комбинировать с проекциями); что произойдёт при сохранении 3D-графика в PNG (схлопнется в текущий ракурс — вращение возможно только интерактивно).

In [ ]:
# --- Figure3D ---
class Figure3D:
    def __init__(self, figsize=(8, 6), **kw):
        self.fig = plt.figure(figsize=figsize, **kw)
        self.ax = self.fig.add_subplot(111, projection="3d")

    def helix(self, n=200, **kw):
        t = np.linspace(0, 6 * np.pi, n)
        self.ax.plot(np.cos(t), np.sin(t), t, **kw)
        return self

    def cloud(self, n=500, **kw):
        rng = np.random.default_rng(42)
        x = rng.normal(0, 1, n)
        y = rng.normal(0, 1, n)
        z = x ** 2 + y ** 2 + rng.normal(0, 0.2, n)
        self.ax.scatter(x, y, z, c=z, cmap="viridis", s=20, **kw)
        return self

    def view(self, elev=30, azim=-60):
        self.ax.view_init(elev=elev, azim=azim)
        return self

    def savefig(self, path, **kw):
        self.fig.savefig(path, **kw)
        return self

# --- демонстрация: спираль + облако ---
(
    Figure3D(figsize=(9, 7))
    .helix(n=300, color="crimson", linewidth=2)
    .cloud(n=500)
    .view(20, -45)
    .savefig("scene3d.png", dpi=150, bbox_inches="tight")
)

# --- 3D-облако возраста/веса/тренировок (реалистичный пример) ---
np.random.seed(135)
n = 400
age = np.random.randint(18, 65, size=n)
weight = 60 + 0.3 * age + np.random.normal(0, 10, n)
workouts = np.clip(6 - 0.05 * age + np.random.normal(0, 1.2, n), 0, 7)

fig = plt.figure(figsize=(9, 7), layout="constrained")
ax = fig.add_subplot(111, projection="3d")
sc = ax.scatter(age, weight, workouts, c=workouts,
                cmap="viridis", s=25, depthshade=True)
ax.set_xlabel("Возраст")
ax.set_ylabel("Вес, кг")
ax.set_zlabel("Тренировок/нед")
ax.set_title("3D-связь возраста, веса и тренировок")
ax.view_init(elev=20, azim=-45)
fig.colorbar(sc, ax=ax, shrink=0.6, label="Тренировок/нед")

# Комментарии:
# Класс-обёртка переиспользует boilerplate (fig + Axes3D) и даёт
# fluent API: цепочка helix().cloud().view().savefig() читается как
# сценарий.
# projection="3d" в subplot_kw/add_subplot возвращает Axes3D —
# подкласс Axes с 3D-трансформациями и методами (plot3D, scatter,
# view_init). Это ООП-расширение, а не "магия".
# view_init(elev, azim): elev — угол подъёма 0–90° (0 — вид сбоку,
# 90 — сверху), azim — поворот вокруг вертикали 0–360°. Задаёт
# статический ракурс для сохранения.
# depthshade=True (по умолчанию) затемняет точки по глубине —
# иллюзия объёма; можно выключить для "плоского" вида.
# 3D читать сложнее 2D: перспективные искажения, перекрытия,
# неоднозначность относительного положения. Для презентации
# комбинируют с 2D-проекциями (scatter по парам переменных) —
# 3D даёт обзор, 2D — точные сравнения.
# Сохранение в PNG "замораживает" текущий ракурс: вращение мышью
# доступно только интерактивно (в Jupyter — %matplotlib widget/
# notebook, в скрипте — plt.show()). Для воспроизводимости фиксируйте
# view_init и сохраняйте.

plt.show()
print()



**Задача 36 (3D-поверхность с проекцией контуров).** Фитнес-клуб моделирует результат клиента как функцию числа тренировок и калорийности питания. Визуализируйте поверхность `Z = np.sin(np.sqrt(X**2 + Y**2))` (волновая — имитация колебаний результата). Постройте два подграфика в 3D: (а) `ax.plot_surface(X, Y, Z, cmap="viridis", linewidth=0, antialiased=True, rstride=2, cstride=2, alpha=0.9)` + `ax.contour(X, Y, Z, zdir="z", offset=-1.5, levels=15, cmap="viridis")` — проекция контуров на «пол»; (б) та же поверхность, но `zdir="x", offset=-5` — проекция на боковую «стену». Настройте `ax.set_box_aspect((1, 1, 0.6))`. Ожидаемый вывод: поверхность с контурами на «полу» (или «стене»). В комментариях объясните: зачем `antialiased=True` (сглаживает грани — без него «рваный» вид); как `rstride=2, cstride=2` уменьшают число полигонов в 4 раза (ускоряет отрисовку ценой детализации); что делает `zdir`/`offset` (выбор плоскости проекции и её сдвиг); почему `set_box_aspect((1, 1, 0.6))` улучшает восприятие (стандартная 1:1:1 «растягивает» Z); что произойдёт при `plot_surface` 1000×1000 (медленно — стоит `rstride=4`); как комбинация поверхности + контуров даёт максимум информации (форма + числовые значения).

In [ ]:
# --- сетка для поверхности ---
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))

fig = plt.figure(figsize=(14, 6), layout="constrained")

# --- (а) контуры на "пол" (zdir="z") ---
ax_a = fig.add_subplot(1, 2, 1, projection="3d")
ax_a.plot_surface(
    X, Y, Z, cmap="viridis", linewidth=0,
    antialiased=True, rstride=2, cstride=2, alpha=0.9,
)
ax_a.contour(
    X, Y, Z, zdir="z", offset=-1.5,
    levels=15, cmap="viridis",
)
ax_a.set_box_aspect((1, 1, 0.6))
ax_a.set_title('Контуры на "полу" (zdir="z")')
ax_a.set_xlabel("Тренировки")
ax_a.set_ylabel("Калорийность")
ax_a.set_zlabel("Результат")
ax_a.view_init(elev=30, azim=-60)

# --- (б) контуры на боковую "стену" (zdir="x") ---
ax_b = fig.add_subplot(1, 2, 2, projection="3d")
ax_b.plot_surface(
    X, Y, Z, cmap="viridis", linewidth=0,
    antialiased=True, rstride=2, cstride=2, alpha=0.9,
)
ax_b.contour(
    X, Y, Z, zdir="x", offset=-5,
    levels=15, cmap="viridis",
)
ax_b.set_box_aspect((1, 1, 0.6))
ax_b.set_title('Контуры на "стене" (zdir="x")')
ax_b.set_xlabel("Тренировки")
ax_b.set_ylabel("Калорийность")
ax_b.set_zlabel("Результат")
ax_b.view_init(elev=30, azim=-60)

fig.savefig("surface_contours.png", dpi=150, bbox_inches="tight")

# --- демонстрация "медленной" сетки 1000x1000 ---
fig2 = plt.figure(figsize=(8, 6), layout="constrained")
ax2 = fig2.add_subplot(111, projection="3d")
x2 = np.linspace(-5, 5, 1000)
y2 = np.linspace(-5, 5, 1000)
X2, Y2 = np.meshgrid(x2, y2)
Z2 = np.sin(np.sqrt(X2**2 + Y2**2))
ax2.plot_surface(X2, Y2, Z2, cmap="viridis", rstride=4, cstride=4,
                 linewidth=0, antialiased=False)
ax2.set_title("1000×1000 с rstride=cstride=4")

# Комментарии:
# antialiased=True сглаживает грани полигонов (субпиксельно), убирая
# "лесенки" по краям; без него поверхность выглядит рваной. Стоимость —
# немного больше времени отрисовки, но визуально критично.
# rstride=2, cstride=2 — рисовать каждый 2-й узел сетки по строкам/
# столбцам. Число полигонов падает в 2×2=4 раза → быстрее отрисовка;
# детализация слегка теряется (для гладких функций — незаметно).
# zdir/offset: zdir выбирает плоскость проекции контуров:
#   "z" — горизонтальный "пол" (offset — координата Z),
#   "x" — вертикальная "стена" (offset — координата X),
#   "y" — другая стена.
# offset — сдвиг этой плоскости (например, -1.5 ниже минимума Z,
# чтобы контуры не пересекались с поверхностью).
# set_box_aspect((1, 1, 0.6)) — соотношение сторон 3D-куба.
# По умолчанию 1:1:1, что визуально "растягивает" Z-ось; 0.6 делает
# её ниже — восприятие формы становится адекватным.
# plot_surface на сетке 1000×1000 = 1 000 000 полигонов — медленно
# (секунды отрисовки). rstride=4/cstride=4 → ~62 500 полигонов
# (16× меньше) при сохранении узнаваемой формы.
# Поверхность + контуры = форма (3D) + числовые значения (2D-изолинии):
# видно и "холмы/ямы", и точные уровни (levels=15) без чтения по Z-оси.

plt.show()
print()



**Задача 37 (confusion matrix из sklearn.metrics).** Фитнес-клуб обучил классификатор, который предсказывает статус клиента (3 класса: «активный», «на грани», «ушедший»). Матрица ошибок покажет, где модель путается. Постройте `cm = confusion_matrix(y_test, y_pred)` и `cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)`, визуализируйте через `imshow(cm, cmap="Blues")` с аннотациями `for i, j in np.ndindex(cm.shape): ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")`. Ожидаемый вывод: две матрицы — абсолютная и нормированная. В комментариях объясните: почему строки — истина, столбцы — предсказание (не путайте — это критично); зачем нормализация при дисбалансе (если «активный» 70%, абсолютные числа вводят в заблуждение); как `ax.text(j, i)` соответствует `imshow` (координаты `(x, y)`, где `x` — столбец); что показывает диагональ (правильные предсказания — чем ярче, тем лучше); как добавить accuracy, precision, recall, F1 в заголовок (`f"Acc={acc:.2f}, F1={f1:.2f}"`); что произойдёт при 10 классах (аннотации станут мелкими — используйте `plt.cm.Blues` без текста).

**Задача 38 (ROC, PR и калибровочная кривая).** Фитнес-клуб обучает модель предсказания оттока клиента на сильно несбалансированных данных (90% остаются, 10% уходят). ROC может «обмануть», PR честнее, калибровка показывает, можно ли доверять вероятностям. Обучите `LogisticRegression` на `make_classification(weights=[0.90, 0.10])`, получите `y_proba = model.predict_proba(X_test)[:, 1]`. Постройте три подграфика: (а) ROC (`fpr, tpr, _ = roc_curve(...); auc = roc_auc_score(...)`); (б) PR (`precision, recall, _ = precision_recall_curve(...); ap = average_precision_score(...)` с `ax.axhline(y_test.mean(), color="gray", linestyle="--", label="Baseline")`); (в) калибровка (`prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)` с диагональю `ax.plot([0, 1], [0, 1], "k--")`). Ожидаемый вывод: три кривые с метриками в легенде. В комментариях объясните: что такое TPR и FPR (чувствительность и доля ложных срабатываний); почему PR лучше ROC при дисбалансе (ROC «смотрит» и на TN, которых 90%, и картина кажется хорошей даже для плохой модели); что означает AUC = 0.5 (случайное угадывание — диагональ); как калибровочная кривая выявляет «самоуверенную» модель (точки ниже диагонали — предсказанные вероятности выше реальных); как `average_precision_score` связан с площадью PR; что произойдёт при перевёрнутых метках классов (AUC станет 1 - AUC, ROC «перевернётся»).

**Задача 39 (learning curves и диагностика переобучения).** Фитнес-клуб обучает модель прогноза посещаемости на исторических данных, но нужно понять, хватит ли данных и не переобучается ли модель. Кривые обучения показывают, как accuracy меняется с ростом обучающей выборки. Обучите три модели: простая `DecisionTree(max_depth=3)` (недообучение), сложная `DecisionTree(max_depth=20)` (переобучение), `RandomForest` (хорошая). Для каждой вызовите `learning_curve(model, X, y, train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring="accuracy", n_jobs=-1)` и постройте train/val с `fill_between(mean ± std)`. Ожидаемый вывод: 3 learning curves с диагнозами. В комментариях объясните: как диагностировать **переобучение** (train высоко, val низко — большой gap); **недообучение** (обе кривые низко — модель слишком простая); «хорошую» модель (кривые сходятся около 0.9+); что показывает заливка `fill_between` (стандартное отклонение по фолдам — стабильность); когда нужно собирать больше данных (обе кривые ещё растут); что произойдёт при `train_sizes=[1.0]` (одна точка — график неинформативен); почему `cv=5` (5 фолдов дают разумный баланс скорости и стабильности).

**Задача 40 (training curves и ранняя остановка).** Фитнес-клуб обучает нейросеть для прогноза посещаемости. По кривым обучения видно, когда модель начинает переобучаться. Смоделируйте историю 50 эпох: `train_loss = 1.5 * np.exp(-epochs/15) + 0.1 + np.random.randn(50)*0.02`, `val_loss = 1.5 * np.exp(-epochs/15) + 0.15 + np.random.randn(50)*0.03`, начиная с 25-й эпохи добавьте `val_loss[25:] += (epochs[25:] - 25) * 0.01` (имитация переобучения). Постройте два подграфика: loss и accuracy (`1 - loss/2`). Найдите `best_epoch = np.argmin(val_loss) + 1`, добавьте `ax.axvline(best_epoch, color="green", linestyle="--", label=f"Best = {best_epoch}")` и `ax.annotate("Начало переобучения", xy=(best_epoch, val_loss[best_epoch-1]), xytext=(best_epoch+5, val_loss[best_epoch-1]+0.3), arrowprops=dict(arrowstyle="->", color="green"))`. Ожидаемый вывод: loss падает, val растёт после 25-й эпохи; best-эпоха отмечена. В комментариях объясните: почему loss падает, а val растёт (переобучение — модель «запоминает» шум); что такое early stopping (остановка на best_epoch — оптимум валидации); как `argmin` находит лучшую эпоху и почему `+ 1` (эпохи начинаются с 1); что делать при взрыве loss (уменьшить learning rate, добавить регуляризацию); зачем логировать метрики в `history = {"train_loss": [], "val_loss": [], ...}` и строить после (`history["train_loss"].append(...)`); что произойдёт при `argmin` на пустом массиве (`ValueError`).

**Задача 41 (feature importance: MDI vs permutation).** Фитнес-клуб обучил RandomForest для прогноза оттока, и нужно понять, какие признаки важнее всего — это поможет в удержании клиентов. У `RandomForest` есть встроенная MDI-importance (`model.feature_importances_`), но она завышает важность категориальных признаков с высокой кардинальностью (например, «ФИО тренера» с 100 уникальными значениями). Permutation importance надёжнее — измеряет падение метрики при перемешивании. Постройте `barh` с двумя сериями: MDI (`pd.Series(model.feature_importances_, index=names).sort_values()`) и permutation (`permutation_importance(model, X, y, n_repeats=10).importances_mean`). Ожидаемый вывод: два barh рядом; MDI и permutation могут расходиться. В комментариях объясните: чем MDI отличается от permutation (MDI — по «нечистоте» узлов; permutation — по падению метрики на holdout); зачем `n_repeats=10` (усреднение по случайным перемешиваниям — уменьшает дисперсию); почему `barh` удобнее для длинных имён признаков (`"количество_тренировок_в_неделю"` не влезает под `bar`); как `plt.cm.viridis(imp / imp.max())` создаёт градиент (важные — яркие, неважные — тёмные); что показывает «важность = 0.15» (признак объясняет 15% «важности» — но это не 15% точности); какие альтернативы (SHAP, `permutation_importance` с `scoring="roc_auc"` вместо accuracy).

**Задача 42 (heatmap z-score с кастомной аннотацией).** Фитнес-клуб анализирует матрицу «филиал × месяц» с числом новых клиентов, но масштабы разные (флагманский филиал — сотни, новый филиал — десятки), поэтому нужно стандартизовать данные. Вычислите `z = (df - df.mean()) / df.std()` (по столбцам), постройте `im = ax.imshow(z, cmap="RdBu_r", vmin=-3, vmax=3)` с `cbar = fig.colorbar(im); cbar.ax.axhline(0, color="black")`. Для ячеек с `|z| > 2` (статистические аномалии) добавьте аннотации: `ax.text(j, i, f"{z.iloc[i, j]:.1f}", color="white", fontsize=8, bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.5))`. Ожидаемый вывод: heatmap, где ярко-красные/синие ячейки — аномалии; остальные — нейтральны. В комментариях объясните: зачем стандартизация (сравнение разных масштабов — «аномально высокий» для нового филиала ≠ «аномально высокий» для флагмана); почему `RdBu_r` (инвертированный — привычный «красный = высокий»); как `bbox` делает аннотации читаемыми на цветном фоне; как скрыть аннотации в маленьких ячейках (например, `if abs(z.iloc[i, j]) > 2 and z.shape[0] * z.shape[1] < 100`); зачем `cbar.ax.axhline(0)` (опорная линия нуля); что произойдёт при `vmin=-3, vmax=3` на данных с диапазоном ±10 (всё за пределами обрежется до крайних цветов).

---



## 🔴 Уровень 5. Эксперт (задачи 43–52)

**Задача 43 (Slider с PlotState-датаклассом).** Фитнес-клуб проводит интерактивную демонстрацию для инвесторов: покажите, как меняется прогноз выручки при разных темпах роста цен и сезонных коэффициентах. Интерактивные виджеты превращают статичный график в панель управления. Реализуйте `@dataclass class PlotState: growth: float = 0.05; seasonality: float = 0.2` и два `Slider` — «темп роста» (0.01–0.20) и «сезонность» (0.0–1.0). В callback `def update(_): state.growth = slider_growth.val; state.seasonality = slider_season.val; y = base * (1 + state.growth) ** months * (1 + state.seasonality * np.sin(2 * np.pi * months / 12)); line.set_ydata(y); fig.canvas.draw_idle()`. Расположите слайдеры через `fig.subplots_adjust(bottom=0.25)` и `fig.add_axes([0.15, 0.15, 0.7, 0.03])` (и второй на `[0.15, 0.05, ...]`). Ожидаемый вывод: слайдеры внизу, график перестраивается в реальном времени. В комментариях объясните: зачем `PlotState` (отделить состояние от вида — упрощает тестирование); почему **обязательно** `fig.canvas.draw_idle()` (без него данные обновятся, но график не перерисуется); что произойдёт без `subplots_adjust` (слайдеры перекроют график); как `blitting` ускоряет перерисовку (перерисовывается только линия, а не вся фигура); как `valfmt="%.2f"` форматирует отображение значения (0.05 → "0.05"); что будет при `valstep=0.01` (значения квантуются — удобно для «шага 1%»).

In [ ]:
from matplotlib.widgets import Slider
from dataclasses import dataclass

# --- PlotState ---
@dataclass
class PlotState:
    growth: float = 0.05
    seasonality: float = 0.2

state = PlotState()

# --- базовые данные ---
months = np.arange(0, 24)
base = 300_000  # базовая месячная выручка

def revenue_curve(growth, seasonality):
    return base * (1 + growth) ** months * (
        1 + seasonality * np.sin(2 * np.pi * months / 12)
    )

# --- фигура и линия ---
fig, ax = plt.subplots(figsize=(11, 7))
fig.subplots_adjust(bottom=0.25)

y0 = revenue_curve(state.growth, state.seasonality)
line, = ax.plot(months, y0, color="steelblue", linewidth=2.5,
                marker="o", markersize=5)
ax.set_xlabel("Месяц")
ax.set_ylabel("Прогноз выручки, ₽")
ax.set_title("Интерактивный прогноз выручки")
ax.grid(alpha=0.3)

# --- слайдеры ---
ax_growth = fig.add_axes([0.15, 0.15, 0.7, 0.03])
ax_season = fig.add_axes([0.15, 0.05, 0.7, 0.03])

slider_growth = Slider(
    ax_growth, "Темп роста", 0.01, 0.20,
    valinit=state.growth, valfmt="%.2f", valstep=0.01,
)
slider_season = Slider(
    ax_season, "Сезонность", 0.0, 1.0,
    valinit=state.seasonality, valfmt="%.2f", valstep=0.05,
)

# --- callback ---
def update(_):
    state.growth = slider_growth.val
    state.seasonality = slider_season.val
    y = revenue_curve(state.growth, state.seasonality)
    line.set_ydata(y)
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw_idle()

slider_growth.on_changed(update)
slider_season.on_changed(update)

# --- пример: как программно "подвигать" слайдер ---
slider_growth.set_val(0.10)

# Комментарии:
# PlotState отделяет СОСТОЯНИЕ (growth, seasonality) от вида (line,
# slider). Это упрощает тестирование: логику прогноза можно проверить
# без Matplotlib, а заодно шарить state между виджетами/экспортом.
# fig.canvas.draw_idle() ОБЯЗАТЕЛЕН: set_ydata меняет данные в Line2D,
# но не запускает перерисовку холста. Без него график "застынет"
# визуально до следующего события окна. draw_idle планирует перерисовку
# в ближайший idle (эффективнее draw()).
# Без fig.subplots_adjust(bottom=0.25) слайдеры (add_axes в долях
# фигуры, y=0.05–0.15) наедут на график — верх графика занимает
# почти всю фигуру.
# blitting: вместо полной перерисовки сохраняем "фон" (оси, сетку)
# и перерисовываем только изменившиеся артисты (линию) — быстрее
# в разы. Реализуется через fig.canvas.copy_from_bbox(ax.bbox) +
# restore_region + draw_artist(line) + blit(fig.bbox). Для двух
# слайдеров не критично, но для 30 fps / больших данных — заметно.
# valfmt="%.2f" форматирует значение в подписи слайдера: 0.05 → "0.05".
# valstep=0.01 квантует значения: слайдер "прыгает" по 1% — удобно
# для деловых решений ("темп роста 5%, 6%, ..."), без "магических"
# 0.0537.
# set_val(0.10) программно двигает слайдер — удобно для тестов и
# скриншотов с конкретными значениями.

plt.show()
print()



**Задача 44 (Button, RadioButtons, CheckButtons в одном дашборде).** Фитнес-клуб показывает инвесторам интерактивный дашборд с 4 виджетами: слайдер темпа роста, радио-кнопки для выбора филиала («Центр» / «Север» / «Юг»), чекбоксы для включения/отключения сетки и легенды, кнопка «Сброс». Реализуйте `class InteractivePlot` с методами `_update`, `_change_branch`, `_toggle_grid`, `_reset`. Используйте `RadioButtons.on_clicked(callback)` (принимает метку выбранного варианта), `CheckButtons.on_clicked(callback)` (принимает метку кликнутого флажка), `Button.on_clicked(callback)` (принимает event, но обычно игнорируется). Ожидаемый вывод: 4 виджета работают совместно; кнопка сброса возвращает исходные значения. В комментариях объясните: чем `RadioButtons` отличается от `CheckButtons` (первый — один активный, второй — много независимых); как `Button.on_clicked(callback)` отличается от `Slider.on_changed(callback)` (первый по клику, второй при изменении); зачем хранить ссылки на виджеты (`self.slider = Slider(...)` — иначе GC удалит); как `set_active(index)` у `RadioButtons` программно выбирает вариант; почему `activecolor` улучшает UX (выделяет активный вариант); что произойдёт при двух активных `RadioButtons` (это не «галочки», нельзя выбрать два — попытка вызовет `set_active` и сбросит первый).

In [ ]:
from matplotlib.widgets import Slider, RadioButtons, CheckButtons, Button

# --- данные по филиалам ---
np.random.seed(144)
months = np.arange(0, 24)
base_by_branch = {"Центр": 400_000, "Север": 280_000, "Юг": 220_000}

def revenue_curve(base, growth, seasonality):
    return base * (1 + growth) ** months * (
        1 + seasonality * np.sin(2 * np.pi * months / 12)
    )

# --- InteractivePlot ---
class InteractivePlot:
    def __init__(self):
        self.growth = 0.05
        self.seasonality = 0.2
        self.branch = "Центр"
        self.show_grid = True
        self.show_legend = True

        self.fig, self.ax = plt.subplots(figsize=(11, 8))
        self.fig.subplots_adjust(bottom=0.35, left=0.15, right=0.78)

        self.line, = self.ax.plot(
            months, revenue_curve(base_by_branch[self.branch],
                                  self.growth, self.seasonality),
            color="steelblue", linewidth=2.5, label=self.branch,
        )
        self.ax.set_xlabel("Месяц")
        self.ax.set_ylabel("Прогноз выручки, ₽")
        self.ax.set_title("Интерактивный дашборд филиалов")
        self.ax.grid(self.show_grid, alpha=0.3)
        self.ax.legend(loc="upper left")

        # --- слайдер роста ---
        ax_slider = self.fig.add_axes([0.15, 0.22, 0.5, 0.03])
        self.slider = Slider(ax_slider, "Рост", 0.01, 0.20,
                             valinit=self.growth, valfmt="%.2f",
                             valstep=0.01)
        self.slider.on_changed(self._update)

        # --- радио выбора филиала ---
        ax_radio = self.fig.add_axes([0.82, 0.30, 0.16, 0.20])
        self.radio = RadioButtons(
            ax_radio, ("Центр", "Север", "Юг"),
            active=0, activecolor="orange",
        )
        self.radio.on_clicked(self._change_branch)

        # --- чекбоксы ---
        ax_check = self.fig.add_axes([0.82, 0.10, 0.16, 0.15])
        self.check = CheckButtons(ax_check, ("Сетка", "Легенда"),
                                  (True, True))
        self.check.on_clicked(self._toggle_grid)

        # --- кнопка сброса ---
        ax_reset = self.fig.add_axes([0.15, 0.05, 0.15, 0.05])
        self.btn = Button(ax_reset, "Сброс", hovercolor="0.9")
        self.btn.on_clicked(self._reset)

    def _update(self, _):
        self.growth = self.slider.val
        y = revenue_curve(base_by_branch[self.branch],
                          self.growth, self.seasonality)
        self.line.set_ydata(y)
        self.ax.relim()
        self.ax.autoscale_view()
        self.fig.canvas.draw_idle()

    def _change_branch(self, label):
        self.branch = label
        self.line.set_ydata(revenue_curve(
            base_by_branch[self.branch], self.growth, self.seasonality))
        self.line.set_label(self.branch)
        if self.show_legend:
            self.ax.legend(loc="upper left")
        self.ax.relim()
        self.ax.autoscale_view()
        self.fig.canvas.draw_idle()

    def _toggle_grid(self, label):
        if label == "Сетка":
            self.show_grid = not self.show_grid
            self.ax.grid(self.show_grid, alpha=0.3)
        elif label == "Легенда":
            self.show_legend = not self.show_legend
            if self.show_legend:
                self.ax.legend(loc="upper left")
            else:
                self.ax.legend_.remove()
        self.fig.canvas.draw_idle()

    def _reset(self, _):
        self.slider.reset()          # вернёт valinit
        self.radio.set_active(0)     # выберет "Центр" программно
        # вернём чекбоксы в исходное состояние
        for label, want in (("Сетка", True), ("Легенда", True)):
            cur = label in [t.get_text() for t in
                            self.check.labels
                            if self.check.get_status()[
                                [l.get_text() for l in
                                 self.check.labels].index(label)]]
            # проще: явно привести
        # сброс через повторную инициализацию флагов (у CheckButtons нет reset)
        if not self.show_grid:
            self.check.set_active(0)
        if not self.show_legend:
            self.check.set_active(1)
        self._update(None)

dash = InteractivePlot()

# Комментарии:
# RadioButtons — ВЗАИМОИСКЛЮЧАЮЩИЙ выбор (один активный); CheckButtons —
# НЕЗАВИСИМЫЕ флажки (можно включить несколько). RadioButtons.on_clicked
# отдаёт МЕТКУ выбранного варианта; CheckButtons.on_clicked — метку
# кликнутого; Button.on_clicked — MouseEvent (обычно игнорируется).
# Slider.on_changed срабатывает при КАЖДОМ изменении значения
# (перетаскивание/клик), Button.on_clicked — только по клику.
# Хранить ссылки (self.slider = Slider(...)) ОБЯЗАТЕЛЬНО: иначе
# Python-GC может удалить объект (нет ссылок), и виджет перестанет
# реагировать/исчезнет. Ссылки на self продлевают жизнь.
# RadioButtons.set_active(index) программно выбирает вариант (0 —
# "Центр") — удобно для "сброса" и для скриптов/тестов.
# activecolor="orange" подсвечивает активную радиокнопку — UX:
# пользователь сразу видит, какой филиал выбран.
# RadioButtons не "галочки": два активных одновременно НЕВОЗМОЖНЫ.
# Попытка активировать второй вызовет set_active и снимет первый —
# это by design, не ошибка.
# У CheckButtons нет метода reset(), поэтому сброс реализуют через
# set_active(i) для каждого флажка, у которого статус не совпадает
# с исходным.

plt.show()
print()



**Задача 45 (TextBox и RangeSlider с синхронизацией).** Фитнес-клуб хочет, чтобы администратор мог ввести точный порог цены абонемента через поле ввода или через слайдер — оба варианта должны синхронизироваться. Реализуйте `TextBox` для точного порога (`on_submit` → `slider.set_val(float(text))`), `RangeSlider` для выбора диапазона дат (`ax.set_xlim(*rslider.val)`). Синхронизируйте `Slider` → `TextBox` с флагом `_updating`: `def on_slider(val): if _guard["updating"]: return; _guard["updating"] = True; textbox.set_val(f"{val:.2f}"); _guard["updating"] = False; redraw()`. Обработайте невалидный ввод через `try/except ValueError` с `ax.set_title("Ошибка: введите число")`. Ожидаемый вывод: `TextBox` ↔ `Slider` синхронизированы; `RangeSlider` меняет зум. В комментариях объясните: почему без флага `_updating` возникает бесконечная рекурсия (TextBox → Slider → TextBox → ...); зачем `try/except` для `TextBox` (пользователь может ввести буквы); что делает `RangeSlider.val` (кортеж `(min, max)` из двух значений); как `blitting` ускоряет `RangeSlider` на больших данных (перерисовывается только область внутри зума); что произойдёт при `textbox.set_val` внутри callback без флага (бесконечная рекурсия → переполнение стека); как `slider.set_val(v)` программно устанавливает значение и триггерит `on_changed` (поэтому и нужен флаг).

**Задача 46 (кириллица: FontManager, кэш и fallback).** Фитнес-клуб готовит отчёт на русском, но на Windows без правильного шрифта вместо «Посещаемость по филиалам» появляются «□□□□□□□□□□». Реализуйте `def setup_cyrillic(fonts=("DejaVu Sans", "Liberation Sans", "Arial", "sans-serif")): plt.rcParams.update({"font.family": fonts, "axes.unicode_minus": False, "font.size": 12})`. Проверьте доступность через `{f.name for f in font_manager.fontManager.ttflist}`. Установите свой корпоративный шрифт через `font_manager.fontManager.addfont("/path/to/Corporate.ttf")` и `name = font_manager.FontProperties(fname="/path/to/Corporate.ttf").get_name()`. Ожидаемый вывод: корректные буквы и `−5` (а не `□5`) на осях. В комментариях объясните: почему fallback-цепочка важна (на разных ОС разные шрифты — Linux, macOS, Windows); как `addfont` регистрирует шрифт в Matplotlib (без него `findfont` его не найдёт); что делать, если DejaVu Sans «не виден» (`_load_fontmanager(try_read_cache=False)` — пересобрать кэш); почему `axes.unicode_minus = False` заменяет `−` (U+2212) на `-` (ASCII) — иногда единственный способ на старых шрифтах; как в Jupyter `%matplotlib inline` иногда «ломает» кириллицу (перезапустите ядро + `plt.close("all")`); как проверить, что шрифт содержит кириллицу (`font_manager.findfont("Arial")` и визуально открыть `.ttf`).

**Задача 47 (собственный .mplstyle и комбинирование).** Фитнес-клуб хочет, чтобы все отчёты выглядели в фирменном стиле (оранжевый цвет бренда, шрифт DejaVu Sans, без верхней и правой рамки). Создайте `report.mplstyle` с настройками: `figure.figsize: 10, 6`, `font.family: DejaVu Sans`, `axes.titlesize: 14`, `axes.spines.top: False`, `axes.spines.right: False`, `axes.grid: True`, `grid.alpha: 0.4`, `legend.fontsize: 11`, `axes.unicode_minus: False`, `axes.prop_cycle: cycler('color', ['#FF6B35', '#ff7f0e', ...])` (фирменные цвета). Загрузите `plt.style.use(["seaborn-v0_8", "report.mplstyle"])`. Дополнительно создайте `dark.mplstyle` с `figure.facecolor: black`, `axes.facecolor: black`, `text.color: white`. Постройте три графика в трёх стилях. Ожидаемый вывод: 3 разных внешних вида. В комментариях объясните: как `.mplstyle` работает с `plt.style.use` (парсит файл и применяет к `rcParams`); зачем порядок в списке (второй стиль переопределяет первый); как положить стиль в `~/.config/matplotlib/stylelib/` для загрузки по имени (`plt.style.use("report")`); как комбинировать с `rc_context` для временного применения; что произойдёт при ошибке в `.mplstyle` (`ValueError: Invalid rcparameter` — отладить через `plt.style.use("default")` + `mpl.rcdefaults()`); как использовать `axes.prop_cycle` для автоматической палитры бренда.

**Задача 48 (экспорт: PdfPages, hashlib, логирование).** Фитнес-клуб готовит ежемесячный отчёт: 12 графиков в PDF + отдельные PNG для email-рассылки. Реализуйте `def export_all(fig, name, formats=("png", "pdf", "svg"), dpi=300, log_path=Path("export.log"))`: сохраните в `outputs/{name}.{fmt}`, залогируйте путь, размер и SHA-256 через `logging`. Дополнительно объедините все фигуры в `report.pdf` через `with PdfPages("outputs/report.pdf") as pdf: for fig in figs: pdf.savefig(fig, bbox_inches="tight")`. Ожидаемый вывод: 12+ файлов в `outputs/`; `report.pdf` объединяет все страницы; лог с хэшами. В комментариях объясните: зачем `hashlib.sha256` (кэш CI — не пересобирать отчёт, если входные данные не менялись); как `PdfPages` работает как контекстный менеджер (наследует `AbstractContextManager`, автоматически закрывает файл); почему `logging` лучше `print` (уровни, форматирование, запись в файл); как закрыть фигуры (`plt.close(fig)` или `plt.close("all")` — избежать утечек памяти при 100+ графиках); как `metadata={"Title": "Отчёт за январь", "Author": "Иванов"}` встраивает информацию в PDF; почему `Path` удобнее `os.path` (кросс-платформенные пути, `.mkdir(parents=True, exist_ok=True)`).

**Задача 49 (Seaborn set_theme и relplot с col/row).** Фитнес-клуб анализирует данные о чаевых (встроенный `tips`) как аналог анализа чаевых в смузи-баре клуба. Seaborn — надстройка над Matplotlib, которая автоматизирует «красивые дефолты» и группировку. Выполните `sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)`, загрузите `tips = sns.load_dataset("tips")`, постройте `g = sns.relplot(data=tips, x="total_bill", y="tip", hue="sex", col="time", row="smoker", kind="scatter", height=3, aspect=1.2)`. Настройте `g.set_titles("{col_name} — {row_name}")`, `g.set_axis_labels("Сумма счёта", "Чаевые")`, `g.fig.suptitle(...)`. Ожидаемый вывод: 2×2 FacetGrid (пол × время × курение). В комментариях объясните: как `col`/`row` строят сетку (col — столбцы, row — строки); что такое `FacetGrid` (обёртка над сеткой подграфиков); чем `relplot` (figure-level) отличается от `scatterplot` (axes-level) — первый создаёт сетку, второй рисует на одной оси; как `ax=` встраивает axes-level функцию в `matplotlib`-фигуру; что делает `col_wrap=2` (переносит сетку, если много столбцов); как `g.savefig("grid.png", dpi=200, bbox_inches="tight")` сохраняет всю сетку.

**Задача 50 (Seaborn: распределения — histplot, kdeplot, displot, ecdfplot).** Фитнес-клуб исследует распределение длительности тренировок по типам занятий (йога, кроссфит, бассейн). Seaborn даёт четыре инструмента: `histplot` (гистограмма с KDE), `kdeplot` (сглаженная плотность), `displot` (figure-level сетка), `ecdfplot` (накопленная CDF). Постройте 4 подграфика: `histplot(kde=True)`, `kdeplot(fill=True, common_norm=False)`, `displot(col="class_type", kind="kde")`, `ecdfplot(hue="class_type")`. Ожидаемый вывод: 4 вида визуализации распределений. В комментариях объясните: что делает `kde` (Kernel Density Estimation — сглаженная оценка плотности); зачем `fill=True` (заливка под кривой — читается лучше); чем `displot` отличается от `histplot` (figure-level создаёт сетку автоматически); что такое `ecdf` (эмпирическая CDF — «доля тренировок короче X»); что делает `common_norm=False` (доли внутри групп, а не по всей выборке); что произойдёт при `hue` с 10+ категориями (все кривые перекроются — используйте `displot(col=...)` вместо `kdeplot(hue=...)`).

**Задача 51 (Seaborn: категориальные — boxplot, violinplot, barplot, countplot).** Фитнес-клуб анализирует данные Titanic (встроенный датасет — модель для понимания выживаемости, аналогично анализу удержания клиентов). Постройте 4 подграфика: `boxplot(x="class", y="age", hue="sex")`, `violinplot(x="class", y="age", hue="sex", split=True)` (`split=True` — половина «скрипки» для каждой группы), `barplot(x="class", y="survived", errorbar="ci")` (среднее с CI), `countplot(x="class", hue="sex")` (частоты). Дополнительно `g = sns.catplot(data=titanic, x="class", y="survived", col="sex", kind="bar", errorbar="ci")`. Ожидаемый вывод: 5 видов категориальных графиков. В комментариях объясните: когда `violinplot` лучше `boxplot` (когда важна плотность распределения — мультимодальность); что делает `split=True` (разделяет «скрипку» пополам для двух групп — сравнение распределений напрямую); почему `barplot` показывает **среднее** (не сумму — для суммы используйте `df.groupby(...).agg("sum")`); что кодирует `errorbar="ci"` (доверительный интервал среднего); чем `countplot` отличается от `barplot` (частоты vs значения); как `catplot` (figure-level) vs `barplot` (axes-level) — когда что.

**Задача 52 (Seaborn: heatmap, clustermap и pivot).** Фитнес-клуб анализирует посещаемость по филиалам и месяцам (встроенный `flights` — как аналог). Постройте `p = flights.pivot(index="month", columns="year", values="passengers")`. Seaborn: `sns.heatmap(p, cmap="YlGnBu", annot=True, fmt="d")` (с аннотациями), `sns.heatmap(p.corr(), cmap="RdBu", vmin=-1, vmax=1, center=0)` (корреляции с центром 0), `sns.clustermap(p, cmap="viridis", standard_scale=1)` (иерархическая кластеризация строк и столбцов). Ожидаемый вывод: 3 heatmap-типа; `clustermap` переставляет строки/столбцы по схожести. В комментариях объясните: что делает `annot=True` (добавляет значения в ячейки) и зачем `fmt="d"` (целые числа); зачем `center=0` в корреляциях (белый = ноль, красный = положительный); что такое `clustermap` (кластеризация — похожие месяцы/годы рядом); как `standard_scale=1` нормализует по строкам (сравнение месяцев с разным масштабом); что произойдёт при `annot=True` на 100×100 (мелко — отключите); как сохранить `clustermap` через `g.savefig` (это отдельная фигура, а не `Axes`).

---



## ⚫ Уровень 6. Гуру (задачи 53–60)

**Задача 53 (ReportBuilder — ООП-обёртка для дашбордов).** Фитнес-клуб готовит ежемесячный отчёт из 5–9 графиков. Код легко превращается в «лапшу» из `axes[i, j].plot(...)`. Правильная архитектура отделяет «что рисовать» от «как размещать». Реализуйте `@dataclass class ReportPage: title: str; plot_fn: Callable[[Axes], None]` и `class ReportBuilder` с `build()` (создаёт фигуру и размещает страницы), `save(path, dpi)` (вызывает `build` + `savefig` + `plt.close`). Соберите 5 страниц: динамика посещаемости (line), топ тренеров (bar), распределение длительности (hist), выручка по филиалам (box), heatmap филиалов по месяцам (imshow). Пустые ячейки скройте через `axes.flat[n:].axis("off")`. Ожидаемый вывод: одна фигура с 5 графиками. В комментариях объясните: зачем `ReportPage` (разделяет «что рисовать» и «как размещать» — легко добавить новый график); как `axes.flat[n:]` скрывает пустые ячейки при нечётном числе графиков; зачем `plt.close(fig)` (освободить память при генерации десятков отчётов); как расширить `to_pdf` через `PdfPages` (многостраничный отчёт); как связать с `PlotConfig` (передавать конфигурацию в `__init__`); когда такой класс — овер-инжиниринг (для одного графика проще `plt.subplots`).

In [ ]:
from dataclasses import dataclass
from typing import Callable
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.axes import Axes

# --- данные для 5 страниц ---
np.random.seed(153)
dates = pd.date_range("2024-01-01", periods=90, freq="D")
visits = 150 + np.cumsum(np.random.randint(-5, 6, size=90))
trainers = ["Иванов", "Смирнова", "Кузнецов", "Попова", "Соколов"]
trainer_counts = np.array([180, 150, 130, 110, 90])
durations = np.concatenate([
    np.random.normal(50, 8, 200),
    np.random.normal(70, 10, 150),
    np.random.normal(90, 7, 80),
])
branches = ["Центр", "Север", "Юг", "Запад"]
revenue_by_branch = [np.random.normal(400, 40, 30) for _ in branches]
heat = np.random.randint(50, 250, size=(4, 12))

# --- ReportPage + ReportBuilder ---
@dataclass
class ReportPage:
    title: str
    plot_fn: Callable[[Axes], None]

    def draw(self, ax: Axes):
        self.plot_fn(ax)
        ax.set_title(self.title)

class ReportBuilder:
    def __init__(self, pages, nrows=2, ncols=3, figsize=(16, 10),
                 layout="constrained"):
        self.pages = pages
        self.nrows = nrows
        self.ncols = ncols
        self.figsize = figsize
        self.layout = layout
        self.fig = None
        self.axes = None

    def build(self):
        self.fig, self.axes = plt.subplots(
            self.nrows, self.ncols,
            figsize=self.figsize, layout=self.layout,
        )
        flat = self.axes.flat
        for ax, page in zip(flat, self.pages):
            page.draw(ax)
        n = len(self.pages)
        for ax in self.axes.flat[n:]:
            ax.axis("off")
        return self

    def save(self, path, dpi=150):
        self.build()
        self.fig.savefig(path, dpi=dpi, bbox_inches="tight")
        plt.close(self.fig)
        return self

    def to_pdf(self, path, dpi=150):
        with PdfPages(path) as pdf:
            for page in self.pages:
                fig, ax = plt.subplots(figsize=self.figsize)
                page.draw(ax)
                pdf.savefig(fig, dpi=dpi, bbox_inches="tight")
                plt.close(fig)
        return self

# --- страницы ---
pages = [
    ReportPage("Посещаемость (line)",
               lambda ax: ax.plot(dates, visits, color="steelblue")),
    ReportPage("Топ тренеров (bar)",
               lambda ax: ax.barh(trainers[::-1],
                                  trainer_counts[::-1],
                                  color="seagreen")),
    ReportPage("Длительность (hist)",
               lambda ax: ax.hist(durations, bins=20,
                                  color="orange", edgecolor="black")),
    ReportPage("Выручка по филиалам (box)",
               lambda ax: ax.boxplot(revenue_by_branch,
                                     labels=branches)),
    ReportPage("Heatmap филиал × месяц (imshow)",
               lambda ax: ax.imshow(heat, cmap="viridis",
                                    aspect="auto")),
]

(
    ReportBuilder(pages, nrows=2, ncols=3, figsize=(16, 10))
    .save("monthly_report.png", dpi=150)
)

ReportBuilder(pages, nrows=2, ncols=3).to_pdf("monthly_report.pdf")

# Комментарии:
# ReportPage отделяет "что рисовать" (plot_fn) от "как размещать"
# (build()/сетка). Чтобы добавить новый график, достаточно добавить
# ещё одну ReportPage в список — build подхватит автоматически, без
# правки кода размещения.
# axes.flat[n:].axis("off") скрывает "хвост" — пустые ячейки сетки
# при нечётном числе графиков (5 при 2×3). Визуально чисто, без
# пустых рамок.
# plt.close(fig) освобождает память и фигуры — критично при
# пакетной генерации десятков отчётов (иначе утечка и warning
# "More than 20 figures have been opened").
# to_pdf через PdfPages: каждая страница рисуется в отдельной
# фигуре и сохраняется в многостраничный PDF — удобно для отправки
# управляющей компании.
# Связь с PlotConfig: передавать cfg в __init__ и использовать в
# plt.subplots(figsize=..., dpi=..., facecolor=..., layout=...).
# Конфиг остаётся единственным источником стиля, builder — единственным
# "сборщиком".
# Овер-инжиниринг: для 1–2 графиков класс избыточен — plt.subplots +
# пара вызовов быстрее и понятнее. Класс оправдан при повторяемых
# отчётах, шаблонах и десятках страниц.

plt.show()
print()



**Задача 54 (LearningCurveAnalyzer — автоматическая диагностика).** Фитнес-клуб обучает модели прогноза оттока для 5 филиалов и хочет автоматически получать диагноз «переобучение / недообучение / хорошо». Кривые обучения мощны, но интерпретировать их вручную утомительно. Реализуйте `class LearningCurveAnalyzer` с методами `fit()` (вызывает `learning_curve` и сохраняет `train_mean`, `val_mean`, `train_std`, `val_std`), `diagnose() -> str` (пороги: `gap = train_mean[-1] - val_mean[-1] > 0.1` → overfitting; `val_mean[-1] < 0.7` → underfitting; иначе good), `plot(ax=None)` (рисует с `fill_between`), `compare(others: list)` (сравнивает несколько моделей на одной фигуре). Обучите три модели, сравните, постройте три learning curves с диагнозами в заголовках. Ожидаемый вывод: 3 графика с диагнозами. В комментариях объясните: зачем `diagnose` (автоматизация — не нужно каждый раз смотреть на график); как пороги `gap > 0.1` и `val_mean < 0.7` определяют диагноз (эмпирические — зависят от задачи); зачем `fit()` возвращает `self` (fluent API — `LearningCurveAnalyzer(...).fit().plot()`); как логировать результаты в `pd.DataFrame` для отчёта (`pd.DataFrame({"model": ..., "diagnosis": ..., "val_mean": ...})`); как связать с ML-дашбордом (задача 56) — learning curves как дополнение к метрикам; когда `compare` полезнее одиночного `plot` (сравнение моделей одной серии).


In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# --- данные для оттока ---
X, y = make_classification(
    n_samples=1000, n_features=15, n_informative=8,
    n_redundant=3, flip_y=0.05, random_state=154,
)

class LearningCurveAnalyzer:
    def __init__(self, estimator, name="model", cv=5,
                 train_sizes=np.linspace(0.1, 1.0, 8)):
        self.estimator = estimator
        self.name = name
        self.cv = cv
        self.train_sizes = train_sizes
        self.train_mean = self.val_mean = None
        self.train_std = self.val_std = None

    def fit(self):
        self.sizes, tr, va = learning_curve(
            self.estimator, X, y, cv=self.cv,
            train_sizes=self.train_sizes, scoring="accuracy",
            n_jobs=-1,
        )
        self.train_mean = tr.mean(axis=1)
        self.val_mean = va.mean(axis=1)
        self.train_std = tr.std(axis=1)
        self.val_std = va.std(axis=1)
        return self

    def diagnose(self) -> str:
        gap = self.train_mean[-1] - self.val_mean[-1]
        if gap > 0.1:
            return "overfitting"
        if self.val_mean[-1] < 0.7:
            return "underfitting"
        return "good"

    def plot(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
        ax.plot(self.sizes, self.train_mean, "o-",
                color="steelblue", label="train")
        ax.fill_between(self.sizes,
                        self.train_mean - self.train_std,
                        self.train_mean + self.train_std,
                        alpha=0.2, color="steelblue")
        ax.plot(self.sizes, self.val_mean, "s-",
                color="crimson", label="val")
        ax.fill_between(self.sizes,
                        self.val_mean - self.val_std,
                        self.val_mean + self.val_std,
                        alpha=0.2, color="crimson")
        ax.set_xlabel("Train size")
        ax.set_ylabel("Accuracy")
        ax.set_title(f"{self.name} — {self.diagnose()}")
        ax.legend()
        ax.grid(alpha=0.3)
        return ax

    def compare(self, others):
        fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
        for a in [self] + others:
            a.fit()
            ax.plot(a.sizes, a.val_mean, "o-", label=f"{a.name} ({a.diagnose()})")
        ax.set_xlabel("Train size")
        ax.set_ylabel("Val accuracy")
        ax.set_title("Сравнение learning curves")
        ax.legend()
        ax.grid(alpha=0.3)
        return fig, ax

# --- три модели ---
models = [
    LearningCurveAnalyzer(LogisticRegression(max_iter=1000),
                          name="LogReg"),
    LearningCurveAnalyzer(DecisionTreeClassifier(max_depth=None,
                                                 random_state=0),
                          name="DecisionTree"),
    LearningCurveAnalyzer(RandomForestClassifier(n_estimators=100,
                                                 random_state=0),
                          name="RandomForest"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), layout="constrained")
for a, ax in zip(models, axes):
    a.fit()
    a.plot(ax)

# --- сравнение (val) ---
models[0].compare(models[1:])

# --- лог в DataFrame ---
report = pd.DataFrame([
    {"model": a.name, "diagnosis": a.diagnose(),
     "val_mean": round(a.val_mean[-1], 3),
     "train_mean": round(a.train_mean[-1], 3),
     "gap": round(a.train_mean[-1] - a.val_mean[-1], 3)}
    for a in models
])
print(report.to_string(index=False))

# Комментарии:
# diagnose() автоматизирует интерпретацию learning curve — не нужно
# глазами сравнивать последние точки и оценивать зазор. Пороги
# gap > 0.1 (train сильно выше val — переобучение) и val_mean < 0.7
# (низкое качество даже на train+val — недообучение) ЭМПИРИЧЕСКИЕ:
# зависят от задачи (для 0.95 baseline порог 0.7 бессмыслен).
# fit() возвращает self — fluent API: LearningCurveAnalyzer(...).fit()
# .plot() читается как сценарий.
# Лог в pd.DataFrame удобен для отчёта: модель, диагноз, метрики,
# gap — таблица идёт в управляющую компанию/дашборд без скриншотов.
# Связь с ML-дашбордом (задача 56): learning curves дополняют
# метрики (ROC-AUC, precision) — показывают, что модель упирается
# в данные (overfitting) или в признаки (underfitting).
# compare() полезнее одиночного plot, когда цель — выбрать лучшую
# модель серии: на одной фигуре видно, у кого val выше при большем
# train size, кто стабилизировался, а кто ещё "дышит".

plt.show()
print()


**Задача 55 (Seaborn PairGrid и кастомизация).** Фитнес-клуб анализирует данные клиентов (вес, процент жира, число тренировок, возраст, потребление калорий). `sns.pairplot` строит матрицу рассеяния для всех пар, но `sns.PairGrid` даёт полный контроль: разные типы для верхнего треугольника, нижнего и диагонали. Постройте `sns.pairplot(iris, hue="species", corner=True, diag_kind="kde")` — быстрый вариант, и `g = sns.PairGrid(iris, hue="species", height=2.5); g.map_upper(sns.scatterplot, s=20, alpha=0.6); g.map_lower(sns.kdeplot, fill=True, alpha=0.4); g.map_diag(sns.histplot, kde=True); g.add_legend()` — кастомный. Добавьте аннотацию через `g.axes[0, 0].annotate("Верхний левый", xy=(5.5, 2.5), fontsize=9, color="red")`. Ожидаемый вывод: две grid-визуализации. В комментариях объясните: чем `pairplot` отличается от `PairGrid` (первый — быстрый с дефолтами, второй — гибкий); что делают `map_upper`/`map_lower`/`map_diag` (разные функции для разных частей матрицы); как `g.axes` даёт доступ к Matplotlib-Axes для тонкой настройки (`g.axes[0, 0].set_xlim(...)`); когда `pairplot` слишком медленный (много признаков — `corner=True` убирает половину); как `g.savefig("pairgrid.png", dpi=200, bbox_inches="tight")` — сохраняет всю сетку; что произойдёт при 10+ признаках (матрица 10×10 — нечитаемо, берите топ-5 по важности).

**Задача 56 (MLDashboard — комплексный ML-дашборд).** Фитнес-клуб обучает модель прогноза оттока и хочет «одностраничный отчёт» для операционного штаба: confusion matrix, ROC, PR, feature importance. Реализуйте `class MLDashboard` с методами `_confusion(ax)`, `_roc(ax)`, `_pr(ax)`, `_importance(ax)` и `build()` (создаёт `fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")`, вызывает все 4 метода, добавляет `fig.suptitle(f"ML Dashboard: {type(self.model).__name__}")`). Соберите дашборд, сохраните в `ml_dashboard.png` с `dpi=300` и `bbox_inches="tight"`. Ожидаемый вывод: 4 графика на одной фигуре. В комментариях объясните: как приватные методы (`_confusion`) организуют код (каждый метод — одна ответственность); как `suptitle` не обрезается благодаря `bbox_inches="tight"`; как расширить (`_calibration`, `_lift`, `_learning_curve`); как `dataclass DashboardConfig` улучшил бы API (цвета, размеры шрифтов, dpi); когда такой класс — овер-инжиниринг (для одноразового скрипта проще 4 раза `plt.subplots`); как логировать дашборд в MLflow (сохранять в `mlflow.log_artifact("ml_dashboard.png")`).

**Задача 57 (интерактивный EDA через RadioButtons).** Фитнес-клуб хочет интерактивный инструмент для быстрого знакомства с данными: `RadioButtons` переключает признак (`weight_kg`, `visits`, `duration_min`, `price_rub`), `CheckButtons` — показывать распределение по типу занятий или общее, на графике — гистограмма. Реализуйте `def draw(feature, show_class): ax.cla(); if show_class: for ct, g in df.groupby("class_type"): ax.hist(g[feature].dropna(), alpha=0.5, label=ct, bins=20); ax.legend() else: ax.hist(df[feature].dropna(), bins=20); ax.set_title(feature); fig.canvas.draw_idle()`. Привяжите через `radio.on_clicked(lambda label: draw(label, check.get_status()[0]))` и `check.on_clicked(lambda label: draw(radio.value_selected, not check.get_status()[0]))`. Ожидаемый вывод: 2 виджета + интерактивная гистограмма. В комментариях объясните: зачем `ax.cla()` перед перерисовкой (иначе графики накладываются); как `radio.value_selected` даёт текущий выбор; почему `draw_idle` лучше `draw` (асинхронная перерисовка — не блокирует UI); как `CheckButtons` взаимодействует с `draw` (передаёт текущее состояние); что произойдёт без `cla` (графики накапливаются — визуальная каша); как добавить третий признак (например, `hue` по филиалу) через ещё одни `CheckButtons`.

**Задача 58 (полный пайплайн: read_csv → groupby → savefig + PdfPages).** Фитнес-клуб готовит отчёт автоматически: скрипт читает CSV, строит 4 графика, сохраняет в PDF и PNG, логирует хэши. Реализуйте `def build_report(csv_path, out_dir)`: (1) `df = pd.read_csv(csv_path, parse_dates=["date"], dtype={"visits": "int32", "price_rub": "float32"})`; (2) постройте 4 графика на одной фигуре (line по месяцам, barh по тренерам, heatmap «филиал × месяц», boxplot по типам занятий); (3) сохраните `report.png` (`dpi=300`) и `report.pdf` через `PdfPages`; (4) верните `{"png": sha256, "pdf": sha256}`. Ожидаемый вывод: файлы созданы; хэши возвращены. В комментариях объясните: как `dtype="int32"` экономит память (в 2 раза для целых); зачем `parse_dates` для корректных осей X (иначе строка, а не дата); как `PdfPages` объединяет многостраничный отчёт (`with PdfPages(path) as pdf: pdf.savefig(fig)`); зачем хэши (кэш CI — не пересобирать, если входные данные не менялись); как `Path.mkdir(parents=True, exist_ok=True)` создаёт папку (не падает, если существует); как логировать ошибки чтения (`try/except FileNotFoundError`).

**Задача 59 (полный бизнес-отчёт: 6 графиков + выводы).** Фитнес-клуб готовит квартальный отчёт для управляющей компании. Постройте фигуру 3×2: (0, 0) динамика посещаемости с `rolling(7).mean()`; (0, 1) `barh` по тренерам (число занятий); (1, 0) `boxplot` стоимости тренировок по типам; (1, 1) `scatter` «число тренировок vs процент жира» с `hue=gender`; (2, 0) `heatmap` «филиал × месяц» из `pivot_table`; (2, 1) `pie` доли типов абонементов (топ-5 + «Прочее»). Примените `sns.set_theme(style="whitegrid")` + `setup_cyrillic()`. Сохраните в `quarterly_report.png` (`dpi=300`) и `quarterly_report.pdf`. Сформулируйте **3–5 проверяемых рекомендаций** с числами (например, «Филиал на Севере растёт на 8% в месяц», «Кроссфит даёт 40% выручки»). Ожидаемый вывод: 6 графиков; рекомендации с числами; ограничения анализа. В комментариях объясните: как `ax.axhline(mean, color="red", linestyle="--")` аннотирует среднее; зачем `barh` для длинных ФИО тренеров; как `pivot_table` готовит heatmap (филиал × месяц); как `PlotConfig` централизует настройки (все цвета/шрифты в одном месте); как проверить отсутствие отрицательных цен (`assert (df["price_rub"] >= 0).all()`); какие ограничения анализа (сезонность, праздники, январский наплыв, выбросы).

**Задача 60 (мини-проект: EDAPipeline — полный EDA + PdfPages + интерактивность).** Финальный проект — собрать всё в классе `EDAPipeline`, который делает полный EDA датасета фитнес-клуба и выдаёт отчёт. Реализуйте `class EDAPipeline` с `__init__(df, target=None)`, методами `overview(ax)` (scatter-пары), `correlations(ax)` (heatmap), `distributions(ax)` (violin), `to_pdf(path)` (статический PDF через `PdfPages`), `to_png(path)` (одна PNG-сводка), `interactive()` (RadioButtons-дашборд), `report()` (dict со summary). Загрузите данные клуба (или Titanic), соберите полный отчёт: статические PNG + `report.pdf` + интерактивная фигура. Сформулируйте **5 инсайтов** (например, «клиенты с 3+ тренировками в неделю имеют на 12% меньше отток») и **3 рекомендации** (например, «увеличить число групповых занятий в филиале на Севере в пиковые часы — снижает отток на 8%»). Ожидаемый вывод: PDF-отчёт, PNG, интерактив; инсайты с числами. В комментариях объясните: как `PdfPages` объединяет статические графики (одна страница на график); как отделить статику (`to_pdf`) от интерактива (`interactive`) — интерактив для презентации, статика для отчёта; зачем `target` в конструкторе (для supervised-задач — группировка по целевой переменной); как обрабатывать `NaN` (`fillna`/`dropna` перед визуализацией); какие графики **не** включать в отчёт (перегруженные, 3D-«для красоты», pie с >7 категориями); как проверить воспроизводимость (`np.random.seed(42)` перед каждой итерацией).